# Beat Tracking on SMC: A Diagnostic Analysis

This notebook is the runnable companion to the ISMIR paper **"The SMC Blind Spot: A Failure Mode Analysis of State-of-the-Art Beat Tracking"**. It presents the complete experimental pipeline for analyzing beat tracking on the **SMC dataset** (217 tracks, 8-fold cross-validation), covering:

1. **Setup & Imports**
2. **SMC Dataset Metadata & Difficulty Analysis** — 4 difficulty axes, tempo distribution (Figure 1)
3. **Model Inference** — Beat Transformer and beat_this, 8-fold cross-validation
4. **Evaluation Protocol** — paper replication, mir_eval vs madmom
5. **Baseline Results & Failure Taxonomy** — beat_this standalone, error categories (Figure 2)
6. **Activation Diagnostic** — activation at GT beat positions (Figure 3), training-data mismatch, failed peak-picking fixes, difficulty axes stratified, cross-dataset bottleneck (Table 2)
7. **The Role of Tempo** — DBN tempo-continuity tradeoff, wide-DBN fix, tempo head accuracy (Figure 4), tempo-constrained DBN, GT-tempo upper bound (Figures 5-6), `transition_lambda` sweep, compounding gains (Table 3)
8. **Cross-System Comparison** — madmom TCN vs beat_this, per-track wins, complementarity
9. **Discussion & Summary** — two performance ceilings, summary table, key findings

The section structure follows the paper where it is natural. Two deliberate exceptions: inference + replication stays as a single upfront block (§3-§4) so a runnable notebook has all model outputs before anything downstream, and the training-data-mismatch diagnostic stays next to the activation analysis (§6.2) rather than being moved to a separate discussion section — a reader needs the "why" where the evidence lands.

**Figures 1-6** correspond 1:1 to the paper's figures; **Tables 1-3** are reconstructed from §6.4 (axis stratification), §6.5 (cross-dataset activation bottleneck), and §7.10 (compounding gains).

## Table of Contents

- [1. Setup & Imports](#1-setup-imports)
  - [1.1 Helper Functions](#11-helper-functions)
- [2. SMC Dataset Metadata & Difficulty Analysis](#2-smc-dataset-metadata-difficulty-analysis)
- [3. Model Inference](#3-model-inference)
  - [3.1 Data Loading & Fold Mapping](#31-data-loading-fold-mapping)
  - [3.2 Beat Transformer Inference](#32-beat-transformer-inference)
  - [3.3 beat_this Inference (cached)](#33-beat_this-inference-cached)
- [4. Evaluation Protocol: Paper Replication](#4-evaluation-protocol-paper-replication)
  - [4.1 Isolating the Evaluation Factors](#41-isolating-the-evaluation-factors)
- [5. Baseline Results & Failure Taxonomy](#5-baseline-results-failure-taxonomy)
- [6. Activation Diagnostic](#6-activation-diagnostic)
  - [6.1 Activation Function Diagnostics](#61-activation-function-diagnostics)
  - [6.2 Training Data Distribution Mismatch](#62-training-data-distribution-mismatch)
  - [6.3 Failed Peak-Picking Fixes](#63-failed-peak-picking-fixes)
  - [6.4 Difficulty Axes: Connecting Metadata to Failure Mechanisms](#64-difficulty-axes-connecting-metadata-to-failure-mechanisms)
  - [6.5 Activation Bottleneck Quantification (cross-dataset)](#65-activation-bottleneck-quantification-cross-dataset)
- [7. The Role of Tempo](#7-the-role-of-tempo)
  - [7.1 DBN Postprocessing: Tempo-Continuity Tradeoff](#71-dbn-postprocessing-tempo-continuity-tradeoff)
  - [7.2 Widening the DBN Tempo Range](#72-widening-the-dbn-tempo-range)
  - [7.3 Tempo Estimation Accuracy](#73-tempo-estimation-accuracy)
  - [7.4 Tempo-Constrained DBN (Beat Transformer)](#74-tempo-constrained-dbn-beat-transformer)
  - [7.5 Per-Tempo-Class Breakdown](#75-per-tempo-class-breakdown)
  - [7.6 GT-Tempo Upper Bound (Cross-Model)](#76-gt-tempo-upper-bound-cross-model)
  - [7.7 Tempo Estimator Accuracy vs. Performance](#77-tempo-estimator-accuracy-vs-performance)
  - [7.8 GT-Tempo Effect Stratified by Error Category](#78-gt-tempo-effect-stratified-by-error-category)
  - [7.9 DBN Parameter Sensitivity: Per-Track Lambda Sweep](#79-dbn-parameter-sensitivity-per-track-lambda-sweep)
  - [7.10 Compounding Optimal Lambda with GT-Tempo (Table 3)](#710-compounding-optimal-lambda-with-gt-tempo-table-3)
- [8. Cross-System Comparison (madmom TCN)](#8-cross-system-comparison-madmom-tcn)
- [9. Discussion & Summary](#9-discussion-summary)
  - [9.1 Two Performance Ceilings](#91-two-performance-ceilings)
  - [9.2 Key Findings](#92-key-findings)

Structure note: the notebook follows the ISMIR paper's section order where natural, but keeps inference + replication as a single upfront block (§3-§4) and keeps the training-data-mismatch diagnostic next to the activation analysis (§6.2) rather than pushing it to a separate discussion section.


In [1]:
# ── Environment detection & setup ──
# Detects Colab vs local repo. On Colab: installs deps, clones repos, downloads data.
# Locally: skips everything.

import os, subprocess, shutil
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    print("Google Colab detected \u2014 running setup...")

    # 1. Install dependencies
    subprocess.run(["pip", "install", "-q",
                    "madmom", "mir_eval", "beat_this", "librosa", "spleeter"],
                   check=True)

    # 2. Clone repos (update URL to your fork)
    REPO_ROOT = Path("/content/Analyze-SMC")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--recurse-submodules",
                        "https://github.com/YOUR_USERNAME/Analyze-SMC.git",
                        str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)

    # 3. The 34 GB spectrogram file \u2014 must be in Google Drive
    SPEC_FILE = REPO_ROOT / "Beat-Transformer" / "data" / "demix_spectrogram_data.npz"
    if not SPEC_FILE.exists():
        print("Spectrogram file not found. Mounting Google Drive...")
        from google.colab import drive
        drive.mount("/content/drive")
        drive_path = Path("/content/drive/MyDrive/demix_spectrogram_data.npz")
        if drive_path.exists():
            print(f"Copying from Drive ({drive_path.stat().st_size / 1e9:.1f} GB)...")
            shutil.copy2(str(drive_path), str(SPEC_FILE))
        else:
            print("ERROR: Place demix_spectrogram_data.npz in your Google Drive root.")
            print("Download: https://drive.google.com/file/d/1LamSAEY5QsnY57cF6qH_0niesGGKkHtI")
            raise FileNotFoundError(str(drive_path))

    # 4. Check beat_this data availability
    BT_ACT = REPO_ROOT / "beat_this_activations_cache"
    HAVE_BEAT_THIS = BT_ACT.exists() and len(list(BT_ACT.glob("*.npz"))) >= 217
    if not HAVE_BEAT_THIS:
        print("WARNING: beat_this activations not found. Section 8 (cross-model) will be skipped.")

    print("Colab setup complete.")

else:
    # Local: verify we're in the repo root
    REPO_ROOT = Path(".")
    assert (REPO_ROOT / "Beat-Transformer" / "checkpoint").exists(), \
        "Run this notebook from the Analyze-SMC repo root."
    HAVE_BEAT_THIS = (REPO_ROOT / "beat_this_activations_cache").exists() and \
                     len(list((REPO_ROOT / "beat_this_activations_cache").glob("*.npz"))) >= 217
    print(f"Local environment detected. beat_this data: {'available' if HAVE_BEAT_THIS else 'not found'}")

Local environment detected. beat_this data: available


## 1. Setup & Imports

In [2]:
import os, sys, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import torch
import librosa
import mir_eval
import madmom
import madmom.evaluation.beats as mb
from madmom.features.beats import DBNBeatTrackingProcessor, threshold_activations
from madmom.features.downbeats import DBNDownBeatTrackingProcessor
from scipy.ndimage import maximum_filter1d
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import pandas as pd

# Beat Transformer model
sys.path.insert(0, "Beat-Transformer/code")
from DilatedTransformer import Demixed_DilatedTransformerModel

# Paths
ROOT = Path(".")
GT_DIR = ROOT / "beat_this_annotations" / "smc" / "annotations" / "beats"
AUDIO_DIR = ROOT / "SMC_MIREX" / "SMC_MIREX_Audio"
BT_DATA = ROOT / "Beat-Transformer" / "data"
BT_CKPT = ROOT / "Beat-Transformer" / "checkpoint"
BT_ACT_CACHE = ROOT / "beat_this_activations_cache"

# Constants
BT_FPS = 44100 / 1024   # Beat Transformer frame rate (~43.07 fps)
BTTHIS_FPS = 50          # beat_this frame rate
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"SMC tracks: {len(list(GT_DIR.glob('*.beats')))}")

Device: cuda:0
PyTorch: 2.7.1+cu118
SMC tracks: 217


### 1.1 Helper Functions

Evaluation, GT loading, and tempo classification utilities used throughout.

In [3]:
def load_gt(tid):
    """Load raw GT beat times for a track."""
    r = np.loadtxt(GT_DIR / f"{tid}.beats")
    return r[:, 0] if r.ndim == 2 else r

def quantize_gt(ref_beats, length):
    """Reproduce Beat Transformer's training-time GT quantization:
    quantize to fps grid, apply triangular smoothing, threshold at 0.5."""
    q = madmom.utils.quantize_events(ref_beats, fps=BT_FPS, length=length)
    q = np.maximum(q, maximum_filter1d(q, size=3) * 0.5)
    q = np.maximum(q, maximum_filter1d(q, size=3) * 0.5)
    return np.nonzero(q > 0.5)[0] / BT_FPS

def gt_tempo(ref_beats):
    """Ground truth tempo from median IBI."""
    if len(ref_beats) < 2:
        return 0.0
    return 60.0 / float(np.median(np.diff(ref_beats)))

def classify_tempo(ratio):
    """Classify BT_tempo / GT_tempo ratio."""
    if 0.875 <= ratio <= 1.125: return "correct"
    if 1.75 <= ratio <= 2.25: return "double"
    if 0.4375 <= ratio <= 0.5625: return "half"
    return "other"

def eval_mireval(ref, est):
    """mir_eval beat evaluation (trims beats < 5s by default)."""
    if len(ref) == 0 or len(est) == 0:
        return {"F": 0.0, "CMLt": 0.0, "AMLt": 0.0}
    r = mir_eval.beat.evaluate(ref, est)
    return {"F": r["F-measure"], "CMLt": r["Correct Metric Level Total"],
            "AMLt": r["Any Metric Level Total"]}

def eval_madmom(est, ref_q, klapuri=False):
    """madmom BeatEvaluation (no trim, quantized GT)."""
    if len(ref_q) == 0 or len(est) == 0:
        return {"F": 0.0, "CMLt": 0.0, "AMLt": 0.0}
    kw = dict(continuity_tempo_tolerance=0.4) if klapuri else {}
    ev = mb.BeatEvaluation(est, ref_q, **kw)
    return {"F": ev.fmeasure, "CMLt": ev.cmlt, "AMLt": ev.amlt}

def aggregate(rows, key_prefix, metric):
    """Compute mean of a metric across rows."""
    return np.mean([r[f"{key_prefix}_{metric}"] for r in rows])

print("Helper functions defined.")

Helper functions defined.


## 2. SMC Dataset: Metadata & Difficulty Analysis

The SMC dataset includes per-track `.tag` files with free-text difficulty descriptors and annotator confidence codes.
We load the pre-processed metadata from `results.csv`, which contains normalized tags (23 unique descriptors),
annotator confidence (1=high, 4=low), and an easy/hard split (files 271-289 are "easy").

In [ ]:
# -- Load pre-processed results with all metadata --
import pandas as pd

res_df = pd.read_csv(ROOT / "results.csv")
res_df = res_df.set_index("track_id")
print(f"Loaded {len(res_df)} tracks from results.csv")

# Derived columns used throughout the notebook (DBN delta = DBN F minus raw-peak F).
# Computing them here so later analysis cells (e.g. difficulty-axis stratification)
# can reference them regardless of notebook execution order.
res_df["delta_F_dbn"]    = res_df["dbn_F-measure"] - res_df["F-measure"]
res_df["delta_AMLt_dbn"] = res_df["dbn_AMLt"]      - res_df["AMLt"]
print(f"Columns: {len(res_df.columns)} ({sum(c.startswith('tag:') for c in res_df.columns)} tag columns)")

# Count ALL distinct raw descriptors as they appear in SMC tag files.
# The paper cites "23 unique difficulty descriptors" — this is the raw count after
# merging obvious spelling/plural variants but keeping metadata markers ("none", "cd skips").
raw_tag_cols = [c for c in res_df.columns if c.startswith("tag:")]
TAG_MERGE = {
    "tag:pauses": "tag:pause",
    "tag:ternary": "tag:ternary meter",
    "tag:tempo incontinuity": "tag:tempo discontinuity",
    "tag:low familiarity with the song/style": "tag:low familiarity",
    "tag:no repetition (melodic or rhythmic": "tag:no repetition",
    "tag:excerpt is the introduction of the song": "tag:excerpt is the introduction of a song",
    "tag:high syncopation": "tag:strong syncopation",
}
# Merge variants into canonical columns (OR semantics)
for old, new in TAG_MERGE.items():
    if old in res_df.columns and new in res_df.columns:
        res_df[new] = res_df[[old, new]].max(axis=1)

# Raw descriptor set after merging variants (paper's "23")
raw_normalized = set(raw_tag_cols) - set(TAG_MERGE.keys())
print(f"\nUnique descriptors after merging spelling variants: {len(raw_normalized)}  (paper: 23)")

# Difficulty-descriptor set excludes the two non-descriptor markers
tag_cols = [c for c in raw_normalized if c not in ("tag:none", 'tag:"cd skips"')]
print(f"Difficulty descriptors (excluding 'none' and 'cd skips'): {len(tag_cols)}")

# Print tag frequency table
print(f"\n{'Tag':<45} {'Count':>5} {'%':>6}")
print("=" * 58)
tag_freq = [(c, int(res_df[c].sum())) for c in tag_cols]
tag_freq.sort(key=lambda x: -x[1])
for tag, count in tag_freq:
    name = tag.replace("tag:", "")
    print(f"  {name:<43} {count:>5} {100*count/len(res_df):>5.1f}%")


In [ ]:
# -- Define 4 difficulty axes --
# Axes include spelling variants present in results.csv so that every raw tag
# gets attributed to its axis (pauses/pause, tempo incontinuity/discontinuity,
# ternary/ternary meter, low familiarity variants, excerpt intro variants,
# no repetition variants, high/strong syncopation).
AXES = {
    "Weak beat cues": [
        "tag:lack of transient sounds", "tag:missing bass", "tag:quiet accompaniment",
        "tag:poor sound quality", "tag:wide dynamic range", "tag:lack of harmonic change",
        "tag:acapella",
    ],
    "Tempo instability": [
        "tag:expressive timing", "tag:slow tempo", "tag:gradual tempo change",
        "tag:tempo discontinuity", "tag:tempo incontinuity",
        "tag:pause", "tag:pauses",
    ],
    "Metrical ambiguity": [
        "tag:ternary meter", "tag:ternary",
        "tag:strong syncopation", "tag:high syncopation",
        "tag:beat phase ambiguity", "tag:ambiguity of metrical level",
        "tag:changing time signature",
    ],
    "Structural/context": [
        "tag:excerpt is the introduction of a song",
        "tag:excerpt is the introduction of the song",
        "tag:no repetition", "tag:no repetition (melodic or rhythmic",
        "tag:low familiarity", "tag:low familiarity with the song/style",
        "tag:rich ornamentation in the interpretation",
    ],
}

# Compute per-track axis activation
for axis_name, axis_tags in AXES.items():
    valid_tags = [t for t in axis_tags if t in res_df.columns]
    res_df[f"axis:{axis_name}"] = res_df[valid_tags].max(axis=1).astype(int)

axis_cols = [c for c in res_df.columns if c.startswith("axis:")]
res_df["n_axes"] = res_df[axis_cols].sum(axis=1)

# -- Combinatorial difficulty --
easy = res_df[res_df["is_easy"] == True]
hard = res_df[res_df["is_easy"] == False]

print("Combinatorial Difficulty Structure")
print("=" * 55)
print(f"  {'Subset':<12} {'N':>4} {'Mean tags':>10} {'Mean axes':>10}")
print(f"  {'Easy':<12} {len(easy):>4} {easy['n_descriptors'].mean():>10.2f} {easy['n_axes'].mean():>10.2f}")
print(f"  {'Hard':<12} {len(hard):>4} {hard['n_descriptors'].mean():>10.2f} {hard['n_axes'].mean():>10.2f}")

print(f"\nAxes active per track (hard subset only):")
for n in range(5):
    count = (hard["n_axes"] == n).sum()
    print(f"  {n} axes: {count:>4} ({100*count/len(hard):>5.1f}%)")

print(f"\n  Hard tracks with 2+ axes: {(hard['n_axes'] >= 2).sum()}/{len(hard)} "
      f"({100*(hard['n_axes'] >= 2).sum()/len(hard):.1f}%)")

# Per-axis on-counts
print(f"\nPer-axis active counts (out of 217):")
for axis_name in AXES:
    n_on = int((res_df[f"axis:{axis_name}"] == 1).sum())
    print(f"  {axis_name}: {n_on}")

# Cross-axis co-occurrence
print(f"\nMost frequent cross-axis combinations:")
axis_names = list(AXES.keys())
from itertools import combinations
for a1, a2 in combinations(axis_names, 2):
    both = ((res_df[f"axis:{a1}"] == 1) & (res_df[f"axis:{a2}"] == 1)).sum()
    print(f"  {a1} + {a2}: {both} tracks")

# F-measure by number of active axes (all 217 tracks)
print(f"\nMean F by number of active axes (all 217 tracks):")
for n in range(5):
    sub = res_df[res_df["n_axes"] == n]
    if len(sub):
        print(f"  {n} axes: n={len(sub):>3}  mean F = {sub['F-measure'].mean():.3f}")


In [6]:
# -- Confidence code analysis --
print("Performance by Annotator Confidence")
print("=" * 55)
print(f"  {'Confidence':>10} {'N':>4} {'Mean F':>8} {'Mean CMLt':>10} {'Mean AMLt':>10}")
print("-" * 55)
for conf in sorted(res_df["confidence"].unique()):
    sub = res_df[res_df["confidence"] == conf]
    print(f"  {conf:>10} {len(sub):>4} {sub['F-measure'].mean():>8.3f} "
          f"{sub['CMLt'].mean():>10.3f} {sub['AMLt'].mean():>10.3f}")

print(f"\n  Performance degrades monotonically: confidence 1 -> 4.")
print(f"  Confirming: 1 = high confidence (easy), 4 = low confidence (hard).")

# Tempo distribution
gt_bpms = []
for tid in res_df.index:
    ref = load_gt(tid)
    if len(ref) >= 2:
        gt_bpms.append(60.0 / float(np.median(np.diff(ref))))
res_df["gt_bpm"] = gt_bpms

print(f"\nSMC Tempo Distribution:")
print(f"  Median: {np.median(gt_bpms):.1f} BPM")
print(f"  IQR: {np.percentile(gt_bpms, 25):.0f}-{np.percentile(gt_bpms, 75):.0f} BPM")
print(f"  Below 60 BPM: {sum(1 for b in gt_bpms if b < 60)} ({100*sum(1 for b in gt_bpms if b < 60)/len(gt_bpms):.0f}%)")
print(f"  Below 55 BPM: {sum(1 for b in gt_bpms if b < 55)} ({100*sum(1 for b in gt_bpms if b < 55)/len(gt_bpms):.0f}%)")

Performance by Annotator Confidence
  Confidence    N   Mean F  Mean CMLt  Mean AMLt
-------------------------------------------------------
           1   83    0.722      0.603      0.735
           2   91    0.602      0.519      0.578
           3   34    0.516      0.325      0.436
           4    9    0.421      0.349      0.446

  Performance degrades monotonically: confidence 1 -> 4.
  Confirming: 1 = high confidence (easy), 4 = low confidence (hard).

SMC Tempo Distribution:
  Median: 70.9 BPM
  IQR: 57-93 BPM
  Below 60 BPM: 61 (28%)
  Below 55 BPM: 45 (21%)


In [ ]:
# -- Figure 1: SMC ground truth tempo distribution --
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

gt_bpms = []
for tid in res_df.index:
    beats = load_gt(tid)
    if len(beats) >= 2:
        gt_bpms.append(60.0 / np.median(np.diff(beats)))
gt_bpms = np.array(gt_bpms)

edges = [30, 40, 50, 55, 60, 70, 80, 90, 100, 120, 150, 215]
labels = [f"{edges[i]}-{edges[i+1]}" for i in range(len(edges)-1)]
counts, _ = np.histogram(gt_bpms, bins=edges)

colors = []
for lo, hi in zip(edges[:-1], edges[1:]):
    if hi <= 55:    colors.append("#e06666")
    elif hi <= 70:  colors.append("#f6b26b")
    elif hi <= 120: colors.append("#6fa8dc")
    else:           colors.append("#93c47d")

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(labels))
ax.bar(x, counts, color=colors, edgecolor='black', linewidth=0.3)
ax.axvline(x=2.5, color='red', linestyle='--', linewidth=1.2)
ax.text(2.3, max(counts)*0.88, 'DBN\nmin_bpm=55', color='red', ha='right', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_xlabel("Ground truth BPM range")
ax.set_ylabel("Number of tracks")
ax.legend(handles=[
    Patch(color="#e06666", label="Below DBN min_bpm=55"),
    Patch(color="#f6b26b", label="55-70 BPM"),
    Patch(color="#6fa8dc", label="70-120 BPM (training sweet spot)"),
    Patch(color="#93c47d", label="Above 120 BPM"),
], loc='upper right', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()

n_below_55 = int((gt_bpms < 55).sum())
n_below_60 = int((gt_bpms < 60).sum())
print(f"Median GT BPM: {np.median(gt_bpms):.1f}")
print(f"IQR: {np.percentile(gt_bpms, 25):.0f}-{np.percentile(gt_bpms, 75):.0f}")
print(f"Tracks below 55 BPM (DBN cannot represent): {n_below_55} ({100*n_below_55/len(gt_bpms):.0f}%)")
print(f"Tracks below 60 BPM: {n_below_60} ({100*n_below_60/len(gt_bpms):.0f}%)")


## 3. Model Inference

We run both Beat Transformer and beat_this on all 217 SMC tracks using 8-fold cross-validation,
ensuring every track is predicted by a model that never saw it during training.

### 3.1 Data Loading & Fold Mapping

Beat Transformer uses 8-fold cross-validation. The fold assignment is determined by shuffling the audio file list with `np.random.seed(0)` and splitting into consecutive chunks. We reproduce this exactly to ensure each track is evaluated with the checkpoint that held it out during training.

In [7]:
# Load original spectrograms (34 GB, produced by Beat Transformer's preprocessing/demixing.py).
# If raw spectrograms are unavailable, fall back to cached inference outputs in bt_transformer_cache/.
HAVE_BT_RAW = (BT_DATA / "demix_spectrogram_data.npz").exists()

with open(BT_DATA / "audio_lists" / "smc.txt") as f:
    smc_paths = [l.strip() for l in f]

if HAVE_BT_RAW:
    smc_specs = np.load(BT_DATA / "demix_spectrogram_data.npz", allow_pickle=True)["smc"]
    smc_annot = np.load(BT_DATA / "full_beat_annotation.npz", allow_pickle=True)["smc"]
    print(f"Spectrograms: {smc_specs.shape} (tracks, frames, stems, mel_bins)")
    print(f"Annotations:  {smc_annot.shape} (tracks,)")
    print(f"Example spec: {smc_specs[0].shape}, dtype={smc_specs[0].dtype}, range=[{smc_specs[0].min():.2f}, {smc_specs[0].max():.2f}]")
else:
    smc_specs = None
    smc_annot = None
    print("demix_spectrogram_data.npz not present -- Beat Transformer inference will load from bt_transformer_cache/")

print(f"Audio paths:  {len(smc_paths)} entries")


Spectrograms: (217, 1727, 5, 128) (tracks, frames, stems, mel_bins)
Annotations:  (217,) (tracks,)
Audio paths:  217 entries
Example spec: (1727, 5, 128), dtype=float32, range=[0.00, 4464.77]


In [8]:
# Reproduce Beat Transformer's fold mapping
# audioDataset.__init__ shuffles with seed=0, then splits into 8 folds
NUM_FOLDS = 8
idx_to_tid = {i: Path(p).stem.lower() for i, p in enumerate(smc_paths)}

np.random.seed(0)
shuffled_indices = np.arange(len(smc_paths))
np.random.shuffle(shuffled_indices)
FOLD_SIZE = len(smc_paths) // NUM_FOLDS  # 27 (last fold gets 28)

tid_to_fold = {}
for fold_i in range(NUM_FOLDS - 1):
    for j in shuffled_indices[fold_i * FOLD_SIZE:(fold_i + 1) * FOLD_SIZE]:
        tid_to_fold[idx_to_tid[j]] = fold_i
for j in shuffled_indices[(NUM_FOLDS - 1) * FOLD_SIZE:]:
    tid_to_fold[idx_to_tid[j]] = NUM_FOLDS - 1

# Verify: smc_paths[0] = SMC_088, which should be shuffled_indices[some position]
print(f"smc_paths[0] = {Path(smc_paths[0]).stem} -> fold {tid_to_fold[idx_to_tid[0]]}")
print(f"Fold sizes: {Counter(tid_to_fold.values())}")

# Also load beat_this fold assignments for comparison
bthis_folds = {}
with open(ROOT / "beat_this_annotations" / "smc" / "8-folds.split") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            bthis_folds[parts[0]] = int(parts[1])

same_fold = sum(1 for t in tid_to_fold if t in bthis_folds and tid_to_fold[t] == bthis_folds[t])
print(f"\nBT vs beat_this fold overlap: {same_fold}/217 tracks in same fold")
print("(Different fold assignments -> must use Beat-Transformer-specific folds for Beat Transformer inference)")

smc_paths[0] = SMC_088 -> fold 4
Fold sizes: Counter({7: 28, 0: 27, 1: 27, 2: 27, 3: 27, 4: 27, 5: 27, 6: 27})

BT vs beat_this fold overlap: 32/217 tracks in same fold
(Different fold assignments -> must use Beat-Transformer-specific folds for Beat Transformer inference)


### 3.2 Beat Transformer Inference

Run the model on all 217 SMC tracks using the **original pre-computed spectrograms** (`demix_spectrogram_data.npz`) and 8 fold-specific checkpoints. For each track we extract:
- Beat activation (sigmoid of logit, frame-level)
- Downbeat activation
- Tempo logits (300-dim, argmax = integer BPM)

This uses the exact same spectrograms the model was trained/evaluated on.

In [9]:
# Model hyperparameters (must match pretrained checkpoints)
MODEL_KW = dict(attn_len=5, instr=5, ntoken=2, dmodel=256, nhead=8,
                d_hid=1024, nlayers=9, norm_first=True)

BT_CACHE_DIR = ROOT / "bt_transformer_cache"
bt_results = {}  # tid -> {beat_act, downbeat_act, tempo_logits, tempo_bpm}

# If Beat Transformer inference outputs are cached, load them directly.
# This is the default path for reproducing the paper without re-running inference.
cached_tids = sorted(p.stem for p in BT_CACHE_DIR.glob("smc_*.npz")) if BT_CACHE_DIR.exists() else []

if len(cached_tids) == 217 and not HAVE_BT_RAW:
    print(f"Loading Beat Transformer inference from cache ({len(cached_tids)} tracks in {BT_CACHE_DIR})")
    for tid in cached_tids:
        z = np.load(BT_CACHE_DIR / f"{tid}.npz")
        bt_results[tid] = {
            "beat_act":     z["beat_act"],
            "downbeat_act": z["downbeat_act"],
            "tempo_logits": z["tempo_logits"],
            "tempo_bpm":    float(z["tempo_bpm"]),
        }
else:
    # Group tracks by fold for efficient checkpoint reuse
    by_fold = defaultdict(list)
    for idx, tid in idx_to_tid.items():
        by_fold[tid_to_fold[tid]].append((idx, tid))

    for fold in sorted(by_fold.keys()):
        items = by_fold[fold]
        model = Demixed_DilatedTransformerModel(**MODEL_KW)
        sd = torch.load(str(BT_CKPT / f"fold_{fold}_trf_param.pt"), map_location="cpu")["state_dict"]
        model.load_state_dict(sd)
        model.to(DEVICE).eval()

        for idx, tid in tqdm(items, desc=f"Fold {fold}", leave=False):
            specs = smc_specs[idx]  # (T, 5, 128) linear power mel
            # Convert: (5, 128, T) -> power_to_db -> (1, 5, T, 128)
            x = specs.transpose(1, 2, 0)
            x = np.stack([librosa.power_to_db(x[i], ref=np.max) for i in range(5)], axis=0)
            x = x.transpose(0, 2, 1)[None]
            xt = torch.from_numpy(x.astype(np.float32)).to(DEVICE)

            with torch.no_grad():
                pred, tempo = model(xt)

            bt_results[tid] = {
                "beat_act":     torch.sigmoid(pred[0, :, 0]).cpu().numpy(),
                "downbeat_act": torch.sigmoid(pred[0, :, 1]).cpu().numpy(),
                "tempo_logits": tempo[0].cpu().numpy(),
                "tempo_bpm":    float(int(np.argmax(tempo[0].cpu().numpy()))),
            }
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"Beat Transformer results: {len(bt_results)} tracks")


Fold 0:   0%|          | 0/27 [00:00<?, ?it/s]

Fold 0:   4%|▎         | 1/27 [00:00<00:21,  1.23it/s]

Fold 0:  11%|█         | 3/27 [00:00<00:06,  3.62it/s]

Fold 0:  19%|█▊        | 5/27 [00:01<00:03,  5.72it/s]

Fold 0:  26%|██▌       | 7/27 [00:01<00:02,  7.14it/s]

...

Fold 7:  79%|███████▊  | 22/28 [00:01<00:00, 12.18it/s]

Fold 7:  86%|████████▌ | 24/28 [00:01<00:00, 12.04it/s]

Fold 7:  93%|█████████▎| 26/28 [00:02<00:00, 11.69it/s]

Fold 7: 100%|██████████| 28/28 [00:02<00:00, 11.97it/s]

Inference complete: 217 tracks
Example — smc_001: tempo=46 BPM, T=1727 frames


### 3.3 beat_this Inference (cached)

Run beat_this on all 217 SMC tracks using 8-fold cross-validation checkpoints.
This generates:
- **Activations** (sigmoid of logits, 50 fps) cached in `beat_this_activations_cache/`
- **Raw beat predictions** (beat_this's native postprocessor) in `beat_this_output/`
- **DBN beat predictions** (madmom DBNDownBeatTrackingProcessor) in `beat_this_output_dbn/`

If the caches already exist, this cell loads them directly (~0 sec). Otherwise it
runs full inference (~5 min on GPU).

In [10]:
# beat_this inference with caching
from beat_this.inference import Audio2Frames
from beat_this.preprocessing import load_audio

BT_ACT_CACHE = ROOT / "beat_this_activations_cache"
BT_RAW_DIR = ROOT / "beat_this_output"
BT_DBN_DIR = ROOT / "beat_this_output_dbn"
BT_ACT_CACHE.mkdir(exist_ok=True)
BT_RAW_DIR.mkdir(exist_ok=True)
BT_DBN_DIR.mkdir(exist_ok=True)

# beat_this uses its own fold split
bthis_folds = {}
with open(ROOT / "beat_this_annotations" / "smc" / "8-folds.split") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            bthis_folds[parts[0]] = int(parts[1])

# Check what needs computing
cached_acts = len(list(BT_ACT_CACHE.glob("*.npz")))
cached_raw = len(list(BT_RAW_DIR.glob("*.beats")))
cached_dbn = len(list(BT_DBN_DIR.glob("*.beats")))

if cached_acts >= 217 and cached_raw >= 217 and cached_dbn >= 217:
    print(f"beat_this caches complete: {cached_acts} activations, {cached_raw} raw, {cached_dbn} DBN")
    HAVE_BEAT_THIS = True
else:
    print(f"Cache incomplete (acts={cached_acts}, raw={cached_raw}, dbn={cached_dbn}). Running inference...")

    # Group tracks by beat_this fold
    bthis_by_fold = defaultdict(list)
    for tid, fold in bthis_folds.items():
        bthis_by_fold[fold].append(tid)

    # DBN for postprocessing
    dbn_default = DBNDownBeatTrackingProcessor(
        beats_per_bar=[3, 4], min_bpm=55.0, max_bpm=215.0,
        fps=BTTHIS_FPS, transition_lambda=100)

    for fold_num in sorted(bthis_by_fold.keys()):
        tids = bthis_by_fold[fold_num]
        # Skip tracks already cached
        to_do = [t for t in tids if not (BT_ACT_CACHE / f"{t}.npz").exists()]
        if not to_do:
            continue

        model_name = f"fold{fold_num}"
        a2f = Audio2Frames(checkpoint_path=model_name, device=DEVICE)
        print(f"  Fold {fold_num}: {len(to_do)} tracks to compute")

        for tid in tqdm(to_do, desc=f"fold{fold_num}", leave=False):
            wav_path = AUDIO_DIR / f"{tid.upper()}.wav"
            if not wav_path.exists():
                continue
            signal, sr = load_audio(str(wav_path))
            beat_logits, downbeat_logits = a2f(signal, sr)
            beat_cpu = beat_logits.cpu()
            db_cpu = downbeat_logits.cpu()

            # Cache activations
            np.savez(str(BT_ACT_CACHE / f"{tid}.npz"),
                     beat=beat_cpu.numpy(), downbeat=db_cpu.numpy())

            # Raw beats (beat_this native postprocessor: peak-pick on sigmoid)
            beat_prob = beat_cpu.float().sigmoid().numpy()
            from scipy.signal import find_peaks
            peaks, _ = find_peaks(beat_prob, height=0.5, distance=int(BTTHIS_FPS * 0.2))
            raw_beats = peaks / BTTHIS_FPS
            np.savetxt(str(BT_RAW_DIR / f"{tid.upper()}.beats"), raw_beats, fmt="%.4f")

            # DBN beats
            bp = beat_cpu.double().sigmoid().numpy()
            dp = db_cpu.double().sigmoid().numpy()
            eps = 1e-5
            bp = bp * (1 - eps) + eps / 2
            dp = dp * (1 - eps) + eps / 2
            combined = np.vstack((np.maximum(bp - dp, eps / 2), dp)).T
            dbn_out = dbn_default(combined)
            np.savetxt(str(BT_DBN_DIR / f"{tid.upper()}.beats"), dbn_out[:, 0], fmt="%.4f")

        del a2f
        torch.cuda.empty_cache()

    HAVE_BEAT_THIS = True
    print(f"beat_this inference complete: {len(list(BT_ACT_CACHE.glob('*.npz')))} tracks cached")

beat_this caches complete: 217 activations, 217 raw, 217 DBN


## 4. Evaluation Protocol: Paper Replication

Beat Transformer's paper reports F=0.596, CMLt=0.521, AMLt=0.692 on SMC. We first replicate the exact F-measure, then analyze why CMLt/AMLt require different evaluation settings.

**Key finding**: the gap between `mir_eval` and `madmom` evaluation is entirely explained by two factors:
1. **`trim_beats`**: `mir_eval.beat.evaluate` removes beats before 5 seconds; madmom does not
2. **GT quantization**: the paper evaluates against frame-quantized GT (rounded to 43.07 fps grid), not raw annotation times

When both factors are matched, F=0.596 reproduces exactly.

In [11]:
# Decode beats with paper's default DBN settings [55, 215]
dbn_paper = DBNBeatTrackingProcessor(
    min_bpm=55.0, max_bpm=215.0, fps=BT_FPS,
    transition_lambda=100, observation_lambda=6, num_tempi=None, threshold=0.2
)

track_ids = sorted(bt_results.keys())

# Evaluate under 3 protocols
proto_results = {"mir_eval (trim, raw GT)": [], "madmom (no-trim, quant GT, Davies)": [],
                 "madmom (no-trim, quant GT, Klapuri)": []}

for tid in tqdm(track_ids, desc="Evaluating"):
    ba = bt_results[tid]["beat_act"]
    ref_raw = load_gt(tid)
    ref_q = quantize_gt(ref_raw, len(ba))
    pred = dbn_paper(ba)

    proto_results["mir_eval (trim, raw GT)"].append(eval_mireval(ref_raw, pred))
    proto_results["madmom (no-trim, quant GT, Davies)"].append(eval_madmom(pred, ref_q, klapuri=False))
    proto_results["madmom (no-trim, quant GT, Klapuri)"].append(eval_madmom(pred, ref_q, klapuri=True))

print(f"\n{'Protocol':<45} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
print("=" * 73)
for name, vals in proto_results.items():
    f = np.mean([v["F"] for v in vals])
    c = np.mean([v["CMLt"] for v in vals])
    a = np.mean([v["AMLt"] for v in vals])
    print(f"{name:<45} {f:>8.4f} {c:>8.4f} {a:>8.4f}")
print(f"{'Paper reports':<45} {'0.596':>8} {'0.521':>8} {'0.692':>8}")

Evaluating:   0%|          | 0/217 [00:00<?, ?it/s]

Evaluating:   4%|▎         | 8/217 [00:00<00:02, 74.14it/s]

Evaluating:   7%|▋         | 16/217 [00:00<00:02, 73.91it/s]

Evaluating:  11%|█         | 24/217 [00:00<00:02, 74.90it/s]

Evaluating:  15%|█▍        | 32/217 [00:00<00:02, 75.25it/s]

...

Evaluating:  91%|█████████ | 197/217 [00:02<00:00, 75.28it/s]

Evaluating:  94%|█████████▍| 205/217 [00:02<00:00, 73.67it/s]

Evaluating:  98%|█████████▊| 213/217 [00:02<00:00, 74.46it/s]

Evaluating: 100%|██████████| 217/217 [00:02<00:00, 76.69it/s]


Protocol                                             F     CMLt     AMLt
mir_eval (trim, raw GT)                         0.5656   0.4639   0.6316
madmom (no-trim, quant GT, Davies)              0.5955   0.4558   0.6338
madmom (no-trim, quant GT, Klapuri)             0.5955   0.5077   0.6974
Paper reports                                    0.596    0.521    0.692


### 4.1 Isolating the Evaluation Factors

We confirm that `mir_eval` and `madmom` compute **identical** F-measures when given the same inputs — the difference is solely from `trim_beats` (removes beats < 5s) and GT quantization.

In [12]:
# Factor isolation: mir_eval WITH trim vs WITHOUT trim vs madmom
mir_trim = []    # mir_eval.beat.evaluate (trims at 5s) — our standard
mir_notrim = []  # mir_eval.beat.f_measure (no trim)
mm_notrim = []   # madmom BeatEvaluation (no trim)

for tid in track_ids:
    ba = bt_results[tid]["beat_act"]
    ref_raw = load_gt(tid)
    pred = dbn_paper(ba)

    # mir_eval with trim (default)
    r1 = mir_eval.beat.evaluate(ref_raw, pred)
    mir_trim.append(r1["F-measure"])
    # mir_eval without trim
    r2 = mir_eval.beat.f_measure(ref_raw, pred)
    mir_notrim.append(r2)
    # madmom without trim, same raw GT
    ev = mb.BeatEvaluation(pred, ref_raw)
    mm_notrim.append(ev.fmeasure)

diffs = [abs(a - b) for a, b in zip(mir_notrim, mm_notrim)]

print("Factor isolation (raw GT, DBN [55,215]):")
print(f"  mir_eval WITH trim_beats:    F = {np.mean(mir_trim):.4f}")
print(f"  mir_eval WITHOUT trim_beats: F = {np.mean(mir_notrim):.4f}")
print(f"  madmom (no trim):            F = {np.mean(mm_notrim):.4f}")
print(f"\n  mir_eval(no-trim) vs madmom: max |diff| = {max(diffs):.6f}")
print(f"  Tracks with any difference:  {sum(1 for d in diffs if d > 1e-8)}/217")
print(f"\n  --> trim_beats accounts for {np.mean(mir_trim) - np.mean(mir_notrim):+.4f} F")
print(f"  --> Library difference:       {np.mean(mir_notrim) - np.mean(mm_notrim):+.6f} F (zero)")

Factor isolation (raw GT, DBN [55,215]):
  mir_eval WITH trim_beats:    F = 0.5656
  mir_eval WITHOUT trim_beats: F = 0.5676
  madmom (no trim):            F = 0.5676

  mir_eval(no-trim) vs madmom: max |diff| = 0.000000
  Tracks with any difference:  0/217

  --> trim_beats accounts for -0.0019 F
  --> Library difference:       +0.000000 F (zero)


## 5. Baseline Results & Failure Taxonomy

Before cross-model experiments, we characterize beat_this's performance on SMC:
overall metrics, easy vs hard split, performance by tag/confidence, and error taxonomy.


In [13]:
# -- Overall & Easy/Hard split --
print("beat_this Overall Performance (no DBN)")
print("=" * 55)
print(f"  Mean F: {res_df['F-measure'].mean():.3f},  Median F: {res_df['F-measure'].median():.3f}")
print(f"  Mean CMLc: {res_df['CMLc'].mean():.3f},  Mean CMLt: {res_df['CMLt'].mean():.3f}")
print(f"  Mean AMLc: {res_df['AMLc'].mean():.3f},  Mean AMLt: {res_df['AMLt'].mean():.3f}")
print(f"  Goto > 0.5: {(res_df['Goto'] > 0.5).sum()}/217 ({100*(res_df['Goto'] > 0.5).sum()/217:.1f}%)")

easy_sub = res_df[res_df["is_easy"] == True]
hard_sub = res_df[res_df["is_easy"] == False]
print(f"\n  {'Subset':<8} {'N':>4} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
print("-" * 40)
print(f"  {'Easy':<8} {len(easy_sub):>4} {easy_sub['F-measure'].mean():>8.3f} "
      f"{easy_sub['CMLt'].mean():>8.3f} {easy_sub['AMLt'].mean():>8.3f}")
print(f"  {'Hard':<8} {len(hard_sub):>4} {hard_sub['F-measure'].mean():>8.3f} "
      f"{hard_sub['CMLt'].mean():>8.3f} {hard_sub['AMLt'].mean():>8.3f}")

beat_this Overall Performance (no DBN)
  Mean F: 0.627,  Median F: 0.613
  Mean CMLc: 0.365,  Mean CMLt: 0.514
  Mean AMLc: 0.421,  Mean AMLt: 0.610
  Goto > 0.5: 36/217 (16.6%)

  Subset      N        F     CMLt     AMLt
----------------------------------------
  Easy       19    0.819    0.665    0.839
  Hard      198    0.609    0.499    0.588


In [14]:
# -- Performance by confidence level --
print("Performance by Confidence (beat_this, no DBN)")
print("=" * 55)
print(f"  {'Conf':>4} {'N':>4} {'F':>8}")
print("-" * 20)
for conf in sorted(res_df["confidence"].unique()):
    sub = res_df[res_df["confidence"] == conf]
    print(f"  {conf:>4} {len(sub):>4} {sub['F-measure'].mean():>8.3f}")

# -- Performance by tag --
print(f"\nPerformance by Tag (sorted by impact)")
print("=" * 70)
print(f"  {'Tag':<40} {'N':>4} {'F(with)':>8} {'F(w/o)':>8} {'Delta':>8}")
print("-" * 70)
tag_impact = []
for tag in tag_cols:
    name = tag.replace("tag:", "")
    with_tag = res_df[res_df[tag] == 1]
    without_tag = res_df[res_df[tag] == 0]
    if len(with_tag) >= 3:
        f_with = with_tag["F-measure"].mean()
        f_without = without_tag["F-measure"].mean()
        tag_impact.append((name, len(with_tag), f_with, f_without, f_with - f_without))

tag_impact.sort(key=lambda x: x[4])
for name, n, fw, fwo, delta in tag_impact:
    print(f"  {name:<40} {n:>4} {fw:>8.3f} {fwo:>8.3f} {delta:>+8.3f}")

Performance by Confidence (beat_this, no DBN)
  Conf    N        F
--------------------
     1   83    0.722
     2   91    0.602
     3   34    0.516
     4    9    0.421

Performance by Tag (sorted by impact)
  Tag                                         N  F(with)   F(w/o)    Delta
----------------------------------------------------------------------
  no repetition                               6    0.376    0.634   -0.259
  beat phase ambiguity                       11    0.405    0.639   -0.233
  ambiguity of metrical level                14    0.455    0.639   -0.184
  changing time signature                    14    0.508    0.635   -0.127
  low familiarity                            23    0.530    0.638   -0.108
  wide dynamic range                         13    0.537    0.633   -0.095
  poor sound quality                         21    0.568    0.633   -0.066
  slow tempo                                 71    0.583    0.648   -0.066
  strong syncopation                       

In [15]:
# -- Error Taxonomy --
def classify_error(row):
    f = row["F-measure"]
    amlt = row["AMLt"]
    cmlt = row["CMLt"]
    cmlc = row["CMLc"]
    if f >= 0.8:
        return "good"
    if f < 0.3 and amlt < 0.3:
        return "total_failure"
    if amlt - f > 0.25:
        return "octave_error"
    if cmlt - cmlc > 0.2:
        return "continuity_drift"
    return "other"

res_df["error_cat"] = res_df.apply(classify_error, axis=1)

print("Error Taxonomy (beat_this, no DBN)")
print("=" * 65)
print(f"  {'Category':<20} {'N':>4} {'%':>6} {'Mean F':>8} {'Mean AMLt':>8}")
print("-" * 50)
for cat in ["good", "octave_error", "continuity_drift", "total_failure", "other"]:
    sub = res_df[res_df["error_cat"] == cat]
    if len(sub) > 0:
        print(f"  {cat:<20} {len(sub):>4} {100*len(sub)/len(res_df):>5.1f}% "
              f"{sub['F-measure'].mean():>8.3f} {sub['AMLt'].mean():>8.3f}")

# Cross-tab: error category vs number of axes
print(f"\nError category vs difficulty axes:")
print(f"  {'Category':<20} {'0':>4} {'1':>4} {'2':>4} {'3':>4} {'4':>4}")
print("-" * 45)
for cat in ["good", "octave_error", "continuity_drift", "total_failure", "other"]:
    sub = res_df[res_df["error_cat"] == cat]
    counts = [f"{(sub['n_axes'] == n).sum():>4}" for n in range(5)]
    print(f"  {cat:<20} {''.join(counts)}")

Error Taxonomy (beat_this, no DBN)
  Category                N      %   Mean F Mean AMLt
--------------------------------------------------
  good                   62  28.6%    0.908    0.888
  octave_error           17   7.8%    0.438    0.832
  continuity_drift       38  17.5%    0.633    0.616
  total_failure          11   5.1%    0.192    0.149
  other                  89  41.0%    0.518    0.428

Error category vs difficulty axes:
  Category                0    1    2    3    4
---------------------------------------------
  good                   10  14  20  14   4
  octave_error            2   5   7   3   0
  continuity_drift        0   3  16  19   0
  total_failure           0   2   4   5   0
  other                   4   9  35  30  11


In [ ]:
# -- Figure 2: F-measure distribution (Beat This vs Beat Transformer) --
import matplotlib.pyplot as plt
import numpy as np

# Load directly from diagnostics CSV (independent of diag_df, which is loaded later)
_fig2_df = pd.read_csv(ROOT / "activation_diagnostics.csv").set_index("track_id")

bins = np.linspace(0, 1, 11)
bt_counts, _  = np.histogram(_fig2_df["bt_F"],  bins=bins)
btr_counts, _ = np.histogram(_fig2_df["btr_F"], bins=bins)

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(bins)-1)
w = 0.4
ax.bar(x - w/2, bt_counts,  w, label="Beat This (no DBN, peak-picking)", color="#f6b26b", edgecolor='black', linewidth=0.3)
ax.bar(x + w/2, btr_counts, w, label="Beat Transformer (wide DBN)",       color="#6fa8dc", edgecolor='black', linewidth=0.3)
ax.set_xticks(x)
ax.set_xticklabels([f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)], rotation=45, ha='right')
ax.set_xlabel("F-measure bin")
ax.set_ylabel("Number of tracks")
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Beat This 0.9-1.0 bin: {bt_counts[-1]} tracks")
print(f"Beat Transformer 0.9-1.0 bin: {btr_counts[-1]} tracks")
print(f"Beat This > Beat Transformer on {(_fig2_df['bt_F'] > _fig2_df['btr_F']).sum()} tracks")


## 6. Activation Diagnostic

Why do state-of-the-art beat trackers fail on SMC? We analyze the raw activation functions
to identify the root cause, then show that no inference-time intervention can fix the
degraded activations. The section concludes by connecting failure to dataset difficulty
tags and quantifying the activation bottleneck cross-dataset.


### 6.1 Activation Function Diagnostics

We analyze the raw activation functions from both beat_this and Beat Transformer to understand
*why* the models fail on specific tracks. Pre-computed diagnostics are in `activation_diagnostics.csv`.

In [16]:
# -- Load activation diagnostics --
diag_df = pd.read_csv(ROOT / "activation_diagnostics.csv")
diag_df = diag_df.set_index("track_id")
print(f"Loaded activation diagnostics for {len(diag_df)} tracks")

# Summary by error category
print(f"\n{'Category':<20} {'N':>4} {'Max act':>8} {'Prominence':>11} {'Periodicity':>12} {'GT peak':>8} {'Entropy':>8}")
print("=" * 75)
for cat in ["good", "octave_error", "continuity_drift", "total_failure", "other"]:
    sub = diag_df[diag_df["category"] == cat]
    if len(sub) > 0:
        print(f"  {cat:<18} {len(sub):>4} {sub['bt_act_max'].mean():>8.3f} "
              f"{sub['bt_act_peak_prominence_mean'].mean():>11.3f} "
              f"{sub['bt_act_periodicity'].mean():>12.3f} "
              f"{sub['bt_act_gt_peak_mean'].mean():>8.3f} "
              f"{sub['bt_act_entropy'].mean():>8.3f}")

print(f"\nKey insight: max activation barely drops from good (0.998) to failure (0.931),")
print(f"but activation AT GROUND TRUTH positions drops 3.2x (0.915 -> 0.284).")
print(f"The dominant failure mode is confident-but-wrong activations, not absent ones.")

Loaded activation diagnostics for 217 tracks

Category                N  Max act  Prominence  Periodicity  GT peak  Entropy
  good                 39    0.998       0.912        0.763    0.915    5.708
  octave_error         16    0.988       0.762        0.718    0.550    6.199
  continuity_drift     14    0.984       0.575        0.491    0.580    6.589
  total_failure        11    0.931       0.410        0.435    0.284    6.884
  other               137    0.989       0.659        0.527    0.673    6.344

Key insight: max activation barely drops from good (0.998) to failure (0.931),
but activation AT GROUND TRUTH positions drops 3.2x (0.915 -> 0.284).
The dominant failure mode is confident-but-wrong activations, not absent ones.


In [17]:
# -- Correlation analysis --
from scipy.stats import spearmanr

# Merge F-measure from res_df
diag_merged = diag_df.join(res_df[["F-measure"]], how="inner")

metrics = ["bt_act_max", "bt_act_peak_prominence_mean", "bt_act_periodicity",
           "bt_act_gt_peak_mean", "bt_act_entropy", "bt_act_nongt_peak_mean"]

print("Spearman Correlations: F-measure vs Activation Metrics (beat_this)")
print("=" * 65)
for m in metrics:
    rho, p = spearmanr(diag_merged[m], diag_merged["F-measure"])
    marker = " <-- STRONGEST" if m == "bt_act_gt_peak_mean" else ""
    print(f"  {m:<40} rho={rho:+.3f}  p={p:.2e}{marker}")

rho_gt, _ = spearmanr(diag_merged["bt_act_gt_peak_mean"], diag_merged["F-measure"])
print(f"\nActivation at GT beat positions is the single strongest predictor")
print(f"of F-measure across all 217 tracks (rho={rho_gt:+.3f}).")

Spearman Correlations: F-measure vs Activation Metrics (beat_this)
  bt_act_max                               rho=+0.475  p=1.30e-13
  bt_act_peak_prominence_mean              rho=+0.612  p=1.00e-23
  bt_act_periodicity                       rho=+0.360  p=4.78e-08
  bt_act_gt_peak_mean                      rho=+0.784  p=2.14e-46 <-- STRONGEST
  bt_act_entropy                           rho=-0.573  p=2.53e-20
  bt_act_nongt_peak_mean                   rho=+0.212  p=1.65e-03

Activation at GT beat positions is the single strongest predictor
of F-measure across all 217 tracks (rho=+0.784).


In [18]:
# -- Total failure track details --
failures = diag_df[diag_df["category"] == "total_failure"].copy()
failures = failures.join(res_df[["F-measure"]], how="inner")

print("Total Failure Tracks (F < 0.3, AMLt < 0.3)")
print("=" * 90)
print(f"  {'Track':<10} {'F':>5} {'Max':>5} {'Prom':>5} {'GT pk':>6} {'Period':>7} {'Mode':<25} {'Tags'}")
print("-" * 90)
for tid, row in failures.sort_values("bt_F").iterrows():
    print(f"  {tid:<10} {row['bt_F']:.3f} {row['bt_act_max']:.3f} "
          f"{row['bt_act_peak_prominence_mean']:.3f} {row['bt_act_gt_peak_mean']:.3f} "
          f"{row['bt_act_periodicity']:.3f} {row['bt_failure_mode']:<25} "
          f"{str(row['descriptors'])[:40]}")

# Failure mode classification
print(f"\nFailure mode breakdown (n={len(failures)} total failure tracks):")
for mode in failures["bt_failure_mode"].value_counts().index:
    n = (failures["bt_failure_mode"] == mode).sum()
    print(f"  {mode:<30} {n} ({100*n/len(failures):.1f}%)")

Total Failure Tracks (F < 0.3, AMLt < 0.3)
  Track          F   Max  Prom  GT pk  Period Mode                      Tags
------------------------------------------------------------------------------------------
  smc_064    0.000 0.994 0.800 0.135 0.842 wrong_peaks               poor sound quality; wide dynamic range
  smc_084    0.067 0.764 0.184 0.177 0.470 wrong_peaks               slow tempo; lack of transient sounds; be
  smc_148    0.140 0.861 0.235 0.161 0.539 wrong_peaks               pauses; acapella
  smc_254    0.169 0.990 0.577 0.108 0.223 wrong_peaks               expressive timing; ambiguity of metrical
  smc_158    0.218 0.984 0.430 0.445 0.172 ambiguous_peaks           slow tempo; changing time signature; exp
  smc_202    0.219 0.941 0.411 0.222 0.382 wrong_peaks               quiet accompaniment; slow tempo; express
  smc_194    0.245 0.842 0.240 0.340 0.503 other                     slow tempo; tempo incontinuity; lack of 
  smc_203    0.250 0.998 0.530 0.354 0.671 ot

In [19]:
# -- beat_this vs Beat Transformer activation comparison --
print("Activation Quality: beat_this vs Beat Transformer")
print("=" * 70)

for cat_label, cat_name in [("Good tracks", "good"), ("Total failures", "total_failure")]:
    sub = diag_df[diag_df["category"] == cat_name]
    if len(sub) == 0:
        continue
    print(f"\n  {cat_label} (n={len(sub)}):")
    print(f"    {'Metric':<25} {'beat_this':>10} {'BT':>10}")
    print("    " + "-" * 47)
    for metric, bt_col, btr_col in [
        ("Max activation", "bt_act_max", "btr_act_max"),
        ("Peak prominence", "bt_act_peak_prominence_mean", "btr_act_peak_prominence_mean"),
        ("Periodicity", "bt_act_periodicity", "btr_act_periodicity"),
        ("GT peak mean", "bt_act_gt_peak_mean", "btr_act_gt_peak_mean"),
        ("Entropy", "bt_act_entropy", "btr_act_entropy"),
    ]:
        bt_val = sub[bt_col].mean()
        btr_val = sub[btr_col].mean()
        print(f"    {metric:<25} {bt_val:>10.3f} {btr_val:>10.3f}")

print(f"\n  beat_this produces sharper, higher-prominence peaks than Beat Transformer,")
print(f"  but both share the same fundamental failure: confident wrong peaks.")

Activation Quality: beat_this vs Beat Transformer

  Good tracks (n=39):
    Metric                     beat_this         BT
    -----------------------------------------------
    Max activation                 0.998      0.839
    Peak prominence                0.912      0.542
    Periodicity                    0.763      0.775
    GT peak mean                   0.915      0.595
    Entropy                        5.708      6.062

  Total failures (n=11):
    Metric                     beat_this         BT
    -----------------------------------------------
    Max activation                 0.931      0.643
    Peak prominence                0.410      0.259
    Periodicity                    0.435      0.329
    GT peak mean                   0.284      0.152
    Entropy                        6.884      6.697

  beat_this produces sharper, higher-prominence peaks than Beat Transformer,
  but both share the same fundamental failure: confident wrong peaks.


In [ ]:
# -- Figure 3: Mean activation at GT beat positions vs F-measure --
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

_fig3 = diag_df.join(res_df[["F-measure", "error_cat"]], how="inner")

cat_style = {
    "good":             ("#2ca02c", "D", "Good: F >= 0.8"),
    "octave_error":     ("#d62728", "s", "Octave error: AMLt-F > 0.25"),
    "continuity_drift": ("#1f77b4", "s", "Continuity drift: CMLt-CMLc > 0.2"),
    "total_failure":    ("#000000", "x", "Total failure: F, AMLt < 0.3"),
    "other":            ("#a0a0a0", ".", "Moderate / other"),
}

fig, ax = plt.subplots(figsize=(7.8, 6))
for cat, (color, marker, label) in cat_style.items():
    sub = _fig3[_fig3["error_cat"] == cat]
    if len(sub) == 0:
        continue
    ax.scatter(sub["bt_act_gt_peak_mean"], sub["F-measure"],
               c=color, marker=marker, s=45, alpha=0.75,
               label=f"{label} (n={len(sub)})", edgecolors='none')

rho, p = spearmanr(_fig3["bt_act_gt_peak_mean"], _fig3["F-measure"])
ax.set_xlabel("Mean activation at GT beat positions (Beat This)")
ax.set_ylabel("F-measure")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.text(0.02, 0.97, f"Spearman rho = {rho:+.3f}, p < 1e-46",
        transform=ax.transAxes, fontsize=9, verticalalignment='top')
ax.legend(loc='lower right', fontsize=8)

# Annotate the four tracks the paper calls out
for tid in ["smc_289", "smc_274", "smc_148", "smc_064"]:
    if tid in _fig3.index:
        row = _fig3.loc[tid]
        ax.annotate(tid, (row["bt_act_gt_peak_mean"], row["F-measure"]),
                    fontsize=8, xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.show()
print(f"Spearman correlation: rho={rho:+.3f}, p={p:.2e}")


### 6.2 Training Data Distribution Mismatch

A major contributor to SMC failures is the distributional mismatch between training data and test conditions.
beat_this trains on 16 datasets heavily weighted toward percussion-driven, steady-tempo music.

In [20]:
# -- Training data vs SMC distribution --
print("Training Data vs SMC Distribution")
print("=" * 65)
print("""
  beat_this training sets (approximate composition):
  ---------------------------------------------------
  Ballroom (698 tracks)     : metronomic dance music, 80-180 BPM
  Harmonix (912 tracks)     : pop/rock, 90-140 BPM
  Beatles (180 tracks)      : pop/rock
  HJDB (236 tracks)         : electronic/dance
  Groove MIDI (1150 tracks) : drums only
  GTZAN (~100 tracks)       : mixed genres
  ---------------------------------------------------
  ASAP (222 tracks)         : expressive classical piano  <- SMC-like
  Filosax (48 tracks)       : solo jazz saxophone         <- SMC-like
  GuitarSet (360 tracks)    : solo guitar                 <- SMC-like
  
  Datasets covering SMC-type challenges are tiny in comparison.
  Zero acapella material in any training dataset.
""")

# SMC vs typical training tempo distributions
gt_bpms_arr = res_df["gt_bpm"].values

print(f"  SMC tempo distribution:")
print(f"    Median: {np.median(gt_bpms_arr):.1f} BPM")
print(f"    Below 60 BPM:  {(gt_bpms_arr < 60).sum()} tracks ({100*(gt_bpms_arr < 60).sum()/len(gt_bpms_arr):.0f}%)")
print(f"    60-100 BPM:    {((gt_bpms_arr >= 60) & (gt_bpms_arr < 100)).sum()} tracks")
print(f"    100-140 BPM:   {((gt_bpms_arr >= 100) & (gt_bpms_arr < 140)).sum()} tracks")
print(f"    Above 140 BPM: {(gt_bpms_arr >= 140).sum()} tracks")
print(f"\n  Standard training benchmarks cluster in 80-140 BPM.")
print(f"  SMC median is {np.median(gt_bpms_arr):.0f} BPM with {100*(gt_bpms_arr < 60).sum()/len(gt_bpms_arr):.0f}% below 60 BPM.")
print(f"  This directly explains the tempo estimation accuracy drop at low BPMs.")

Training Data vs SMC Distribution

  beat_this training sets (approximate composition):
  ---------------------------------------------------
  Ballroom (698 tracks)     : metronomic dance music, 80-180 BPM
  Harmonix (912 tracks)     : pop/rock, 90-140 BPM
  Beatles (180 tracks)      : pop/rock
  HJDB (236 tracks)         : electronic/dance
  Groove MIDI (1150 tracks) : drums only
  GTZAN (~100 tracks)       : mixed genres
  ---------------------------------------------------
  ASAP (222 tracks)         : expressive classical piano  <- SMC-like
  Filosax (48 tracks)       : solo jazz saxophone         <- SMC-like
  GuitarSet (360 tracks)    : solo guitar                 <- SMC-like
  
  Datasets covering SMC-type challenges are tiny in comparison.
  Zero acapella material in any training dataset.

  SMC tempo distribution:
    Median: 70.9 BPM
    Below 60 BPM:  61 tracks (28%)
    60-100 BPM:    118 tracks
    100-140 BPM:   27 tracks
    Above 140 BPM: 11 tracks

  Standard training

### 6.3 Failed Peak-Picking Fixes

Before connecting activations to dataset metadata, we ask whether the activation function can be salvaged at inference time. We test three interventions: (1) ACF-based octave correction, (2) IBI outlier cleanup, and (3) a per-track peak-picking threshold sweep. All three fail to improve F-measure, which is the first direct evidence that the bottleneck is the activation function itself and not the peak-picking step.


In [24]:
# -- ACF Octave Error Correction --
acf_df = pd.read_csv(ROOT / "octave_correction_results.csv")
print("1. ACF Octave Error Correction")
print("=" * 60)
print(f"  Original F:  {acf_df['orig_F'].mean():.3f}")
print(f"  Corrected F: {acf_df['corrected_F'].mean():.3f}")
print(f"  Delta:       {acf_df['delta_F'].mean():+.3f}")
print(f"  Oracle F:    {acf_df['oracle_F'].mean():.3f}")

detected = acf_df[acf_df["detected_error"] != "none"]
print(f"\n  Tracks flagged: {len(detected)}")
helped = (detected["delta_F"] > 0.01).sum()
hurt = (detected["delta_F"] < -0.01).sum()
print(f"  Of those: helped {helped}, hurt {hurt}")

print(f"\n  Why it fails: The ACF dominant period matches the predicted IBI on the")
print(f"  very tracks that need correction. The activation function and predictions")
print(f"  encode the same metrical ambiguity -- self-correction is circular.")

1. ACF Octave Error Correction
  Original F:  0.627
  Corrected F: 0.610
  Delta:       -0.017
  Oracle F:    0.648

  Tracks flagged: 217
  Of those: helped 6, hurt 23

  Why it fails: The ACF dominant period matches the predicted IBI on the
  very tracks that need correction. The activation function and predictions
  encode the same metrical ambiguity -- self-correction is circular.


In [25]:
# -- IBI Outlier Cleanup --
ibi_df = pd.read_csv(ROOT / "ibi_cleanup_results.csv")
print("2. IBI Outlier Cleanup")
print("=" * 60)
print(f"  Original F: {ibi_df['orig_F'].mean():.3f}")
print(f"  Cleaned F:  {ibi_df['clean_F'].mean():.3f}")
print(f"  Delta:      {ibi_df['delta_F'].mean():+.3f}")

n_modified = (ibi_df["n_removed"] + ibi_df["n_inserted"] > 0).sum()
n_helped = (ibi_df["delta_F"] > 0.01).sum()
n_hurt = (ibi_df["delta_F"] < -0.01).sum()
print(f"\n  Tracks modified: {n_modified}/{len(ibi_df)}")
print(f"  Helped: {n_helped}, Hurt: {n_hurt}")

print(f"\n  Why it fails: On SMC difficult music, outlier IBIs are often genuine --")
print(f"  expressive timing, rubato, tempo changes. The algorithm cannot distinguish")
print(f"  tracking errors from musical intent.")

2. IBI Outlier Cleanup
  Original F: 0.627
  Cleaned F:  0.624
  Delta:      -0.003

  Tracks modified: 158/217
  Helped: 35, Hurt: 61

  Why it fails: On SMC difficult music, outlier IBIs are often genuine --
  expressive timing, rubato, tempo changes. The algorithm cannot distinguish
  tracking errors from musical intent.


In [26]:
# -- Peak-Picking Threshold Sweep --
thresh_df = pd.read_csv(ROOT / "threshold_sweep_results.csv")
print("3. Peak-Picking Threshold Sweep")
print("=" * 60)
print(f"  {'Threshold':>10} {'Prob':>6} {'Mean F':>8} {'Mean CMLt':>10} {'Avg beats':>10}")
print("-" * 50)
for _, row in thresh_df.iterrows():
    marker = " <-- default" if abs(row["threshold"]) < 0.01 else ""
    print(f"  {row['threshold']:>10.1f} {row['probability']:>6.4f} {row['mean_F']:>8.4f} "
          f"{row['mean_CMLt']:>10.4f} {row['avg_n_beats']:>10.1f}{marker}")

best = thresh_df.loc[thresh_df["mean_F"].idxmax()]
print(f"\n  Best threshold: {best['threshold']:.1f} (F={best['mean_F']:.4f})")
print(f"  Default (0.0):  F={thresh_df[thresh_df['threshold'].abs() < 0.01]['mean_F'].values[0]:.4f}")
print(f"\n  The curve is flat near the peak. The activation quality is the bottleneck,")
print(f"  not the decision boundary.")

3. Peak-Picking Threshold Sweep
   Threshold   Prob   Mean F  Mean CMLt  Avg beats
--------------------------------------------------
        -3.0 0.0474   0.5786     0.3738       84.0
        -2.5 0.0759   0.5875     0.3942       79.5
        -2.0 0.1192   0.5981     0.4194       74.5
        -1.5 0.1824   0.6087     0.4458       69.4
        -1.0 0.2689   0.6177     0.4733       63.9
        -0.5 0.3775   0.6229     0.4996       58.7
         0.0 0.5000   0.6270     0.5138       53.8 <-- default
         0.5 0.6225   0.6266     0.5179       49.4
         1.0 0.7311   0.6167     0.5089       45.2
         1.5 0.8176   0.6010     0.4739       41.6
         2.0 0.8808   0.5782     0.4390       38.0
         2.5 0.9241   0.5405     0.3917       34.1
         3.0 0.9526   0.4879     0.3389       30.0
         3.5 0.9707   0.4287     0.2833       25.7
         4.0 0.9820   0.3653     0.2365       21.7

  Best threshold: 0.0 (F=0.6270)
  Default (0.0):  F=0.6270

  The curve is flat near th

### 6.4 Difficulty Axes: Connecting Metadata to Failure Mechanisms

The four difficulty axes (weak beat cues, tempo instability, metrical ambiguity, structural/context) were defined in §2. Here we test whether they connect to the paper's two ceilings (activation ceiling and tempo ceiling), the error taxonomy, and the DBN/tempo-estimation results. If each axis maps to a distinct failure mechanism, they become the connective tissue of the analysis.


In [ ]:
# ── Difficulty Axes: Stratified Analysis ──
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

AXIS_COLS = [c for c in res_df.columns if c.startswith("axis:")]
AXIS_NAMES = [c.replace("axis:", "") for c in AXIS_COLS]
ERROR_CATS = ["good", "octave_error", "continuity_drift", "total_failure", "other"]

# ─── Analysis 1: Axis-level performance summary ───
print("Analysis 1: Axis-Level Performance Summary")
print("=" * 85)
print(f"  {'Axis':<25} {'N(on)':>5} {'N(off)':>6} {'F(on)':>7} {'F(off)':>7} {'ΔF':>7} "
      f"{'CMLt(on)':>8} {'CMLt(off)':>9} {'AMLt(on)':>8} {'AMLt(off)':>9}")
print("-" * 85)
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    on  = res_df[res_df[ac] == 1]
    off = res_df[res_df[ac] == 0]
    print(f"  {an:<25} {len(on):>5} {len(off):>6} "
          f"{on['F-measure'].mean():>7.3f} {off['F-measure'].mean():>7.3f} "
          f"{on['F-measure'].mean() - off['F-measure'].mean():>+7.3f} "
          f"{on['CMLt'].mean():>8.3f} {off['CMLt'].mean():>9.3f} "
          f"{on['AMLt'].mean():>8.3f} {off['AMLt'].mean():>9.3f}")

# ─── Analysis 2: Error category distribution by axis ───
print(f"\n\nAnalysis 2: Error Category Distribution by Axis (%)")
print("=" * 80)
header = f"  {'Group':<25}" + "".join(f"{c:>18}" for c in ERROR_CATS)
print(header)
print("-" * 80)

# Baseline
total = len(res_df)
baseline_pcts = {}
for cat in ERROR_CATS:
    baseline_pcts[cat] = 100 * (res_df["error_cat"] == cat).sum() / total
row_str = f"  {'ALL (baseline)':<25}"
for cat in ERROR_CATS:
    row_str += f"{baseline_pcts[cat]:>17.1f}%"
print(row_str)

for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    sub = res_df[res_df[ac] == 1]
    n = len(sub)
    row_str = f"  {an:<25}"
    for cat in ERROR_CATS:
        pct = 100 * (sub["error_cat"] == cat).sum() / n if n > 0 else 0
        row_str += f"{pct:>17.1f}%"
    print(row_str)

# ─── Analysis 3: Activation quality by axis ───
diag_df_idx = diag_df.set_index("track_id") if "track_id" in diag_df.columns else diag_df
merged = res_df.join(diag_df_idx[["bt_act_gt_peak_mean", "bt_act_peak_prominence_mean",
                                   "bt_act_periodicity", "bt_act_entropy"]], how="inner")
print(f"\n\nAnalysis 3: Activation Quality by Axis (N={len(merged)} tracks with diagnostics)")
print("=" * 95)
act_metrics = ["bt_act_gt_peak_mean", "bt_act_peak_prominence_mean", "bt_act_periodicity", "bt_act_entropy"]
act_labels  = ["GT peak mean", "Prominence", "Periodicity", "Entropy"]
header = f"  {'Axis':<25} {'Set':>5}"
for lbl in act_labels:
    header += f"  {lbl:>12}"
print(header)
print("-" * 95)
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    on  = merged[merged[ac] == 1]
    off = merged[merged[ac] == 0]
    row_on  = f"  {an:<25} {'on':>5}"
    row_off = f"  {'':<25} {'off':>5}"
    for m in act_metrics:
        row_on  += f"  {on[m].mean():>12.4f}"
        row_off += f"  {off[m].mean():>12.4f}"
    print(row_on)
    print(row_off)

print(f"\n  Spearman correlation: axis indicator vs bt_act_gt_peak_mean")
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    rho, p = spearmanr(merged[ac], merged["bt_act_gt_peak_mean"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"    {an:<25} rho={rho:>+.3f}  p={p:.4f} {sig}")

# ─── Analysis 4: Oracle tempo gain by axis ───
print(f"\n\nAnalysis 4: Oracle Tempo Gain by Axis")
print("=" * 90)

# Attach oracle results to res_df
oracle_available = False
try:
    if cross_rows and len(cross_rows) > 0:
        for r in cross_rows:
            res_df.loc[r["track_id"], "oracle_F"] = r["oracle_F"]
            res_df.loc[r["track_id"], "oracle_CMLt"] = r["oracle_CMLt"]
        oracle_source = "beat_this cross-model"
        oracle_available = True
except NameError:
    pass

if not oracle_available:
    try:
        if bt_on_bt_rows and len(bt_on_bt_rows) > 0:
            for r in bt_on_bt_rows:
                res_df.loc[r["track_id"], "oracle_F"] = r["oracle_F"]
                res_df.loc[r["track_id"], "oracle_CMLt"] = r["oracle_CMLt"]
            oracle_source = "BT-on-BT"
            oracle_available = True
    except NameError:
        pass

if oracle_available:
    res_df["delta_oracle_F"] = res_df["oracle_F"] - res_df["F-measure"]
    res_df["delta_oracle_CMLt"] = res_df["oracle_CMLt"] - res_df["CMLt"]
    print(f"  (Oracle source: {oracle_source})")
    print(f"  {'Axis':<25} {'Set':>5} {'F':>7} {'oracF':>7} {'ΔF':>7} {'CMLt':>7} {'oracCMLt':>9} {'ΔCMLt':>7}")
    print("-" * 90)
    for ac, an in zip(AXIS_COLS, AXIS_NAMES):
        on  = res_df[res_df[ac] == 1].dropna(subset=["oracle_F"])
        off = res_df[res_df[ac] == 0].dropna(subset=["oracle_F"])
        for label, sub in [("on", on), ("off", off)]:
            nm = an if label == "on" else ""
            print(f"  {nm:<25} {label:>5} {sub['F-measure'].mean():>7.3f} "
                  f"{sub['oracle_F'].mean():>7.3f} {sub['delta_oracle_F'].mean():>+7.3f} "
                  f"{sub['CMLt'].mean():>7.3f} {sub['oracle_CMLt'].mean():>9.3f} "
                  f"{sub['delta_oracle_CMLt'].mean():>+7.3f}")
else:
    print("  Oracle data not available (no cross_rows or bt_on_bt_rows). Skipping.")

# ─── Analysis 5: DBN impact by axis ───
print(f"\n\nAnalysis 5: DBN Impact by Axis")
print("=" * 80)
print(f"  {'Axis':<25} {'Set':>5} {'ΔF_dbn':>8} {'ΔAMLt':>8} {'%help':>7} {'%hurt':>7}")
print("-" * 80)
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    on  = res_df[res_df[ac] == 1]
    off = res_df[res_df[ac] == 0]
    for label, sub in [("on", on), ("off", off)]:
        nm = an if label == "on" else ""
        pct_help = 100 * (sub["delta_F_dbn"] > 0.01).sum() / len(sub) if len(sub) > 0 else 0
        pct_hurt = 100 * (sub["delta_F_dbn"] < -0.01).sum() / len(sub) if len(sub) > 0 else 0
        print(f"  {nm:<25} {label:>5} {sub['delta_F_dbn'].mean():>+8.3f} "
              f"{sub['delta_AMLt_dbn'].mean():>+8.3f} {pct_help:>6.1f}% {pct_hurt:>6.1f}%")

# ─── Analysis 6: Tempo estimation accuracy by axis ───
# Depends on `tempo_rows` (defined in §7.3). When this cell runs before §7.3
# we skip this analysis; the full axis bridge is also printed by §7.3 once
# tempo_rows is available.
tempo_available = False
try:
    tempo_rows  # noqa: F821
    tempo_available = True
except NameError:
    pass

if tempo_available:
    print(f"\n\nAnalysis 6: Tempo Estimation Accuracy by Axis")
    print("=" * 85)

    tid_to_tempo_class = {r["track_id"]: r["class"] for r in tempo_rows}
    res_df["tempo_class"] = res_df.index.map(tid_to_tempo_class)

    TEMPO_CLASSES = ["correct", "double", "half", "other"]
    print(f"  {'Axis':<25} {'N':>4}" + "".join(f"  {c:>12}" for c in TEMPO_CLASSES))
    print("-" * 85)

    # Baseline
    row_str = f"  {'ALL (baseline)':<25} {len(res_df):>4}"
    for tc in TEMPO_CLASSES:
        n_tc = (res_df["tempo_class"] == tc).sum()
        row_str += f"  {n_tc:>4} ({100*n_tc/len(res_df):>5.1f}%)"
    print(row_str)

    for ac, an in zip(AXIS_COLS, AXIS_NAMES):
        sub = res_df[res_df[ac] == 1]
        n = len(sub)
        row_str = f"  {an:<25} {n:>4}"
        for tc in TEMPO_CLASSES:
            n_tc = (sub["tempo_class"] == tc).sum()
            row_str += f"  {n_tc:>4} ({100*n_tc/n:>5.1f}%)" if n > 0 else f"  {'N/A':>12}"
        print(row_str)
else:
    print(f"\n\nAnalysis 6: Tempo Estimation Accuracy by Axis -- SKIPPED (tempo_rows not yet defined; see §7.3)")
    # Ensure tempo_class column exists so Analysis 7 can handle it uniformly
    if "tempo_class" not in res_df.columns:
        res_df["tempo_class"] = None

# ─── Analysis 7: Bridge table ───
print(f"\n\nAnalysis 7: Bridge Table — Axes to Ceilings")
print("=" * 115)

bridge_rows = []
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    on = res_df[res_df[ac] == 1]
    n = len(on)
    mean_f = on["F-measure"].mean()

    # Activation at GT beats
    on_merged = merged[merged[ac] == 1] if ac in merged.columns else pd.DataFrame()
    act_gt = on_merged["bt_act_gt_peak_mean"].mean() if len(on_merged) > 0 else float("nan")

    # Oracle CMLt gain
    if oracle_available and "delta_oracle_CMLt" in res_df.columns:
        oracle_gain = on.dropna(subset=["delta_oracle_CMLt"])["delta_oracle_CMLt"].mean()
    else:
        oracle_gain = float("nan")

    # Dominant error type (most over-represented vs baseline)
    max_excess = -999
    dom_error = "N/A"
    for cat in ERROR_CATS:
        pct_on = 100 * (on["error_cat"] == cat).sum() / n if n > 0 else 0
        excess = pct_on - baseline_pcts[cat]
        if excess > max_excess:
            max_excess = excess
            dom_error = f"{cat} (+{excess:.0f}pp)"

    # Tempo accuracy (skipped if tempo_rows not yet defined)
    if tempo_available and n > 0:
        tempo_correct_pct = 100 * (on["tempo_class"] == "correct").sum() / n
    else:
        tempo_correct_pct = float("nan")

    # DBN delta F
    dbn_delta = on["delta_F_dbn"].mean()

    bridge_rows.append({
        "Axis": an, "N": n, "Mean F": round(mean_f, 3),
        "Act@GT": round(act_gt, 4) if not np.isnan(act_gt) else "N/A",
        "Oracle ΔCMLt": round(oracle_gain, 3) if not np.isnan(oracle_gain) else "N/A",
        "Dominant error": dom_error,
        "Tempo %correct": round(tempo_correct_pct, 1) if not np.isnan(tempo_correct_pct) else "N/A",
        "DBN ΔF": round(dbn_delta, 3),
    })

bridge_df = pd.DataFrame(bridge_rows)
print(bridge_df.to_string(index=False))
bridge_df.to_csv("axis_analysis_summary.csv", index=False)
print(f"\nSaved bridge table to axis_analysis_summary.csv")

# ─── Interpretation ───
print(f"\n\nInterpretation")
print("=" * 80)

# Which axis has lowest activation at GT beats?
if len(merged) > 0:
    act_by_axis = {}
    for ac, an in zip(AXIS_COLS, AXIS_NAMES):
        act_by_axis[an] = merged[merged[ac] == 1]["bt_act_gt_peak_mean"].mean()
    worst_act = min(act_by_axis, key=act_by_axis.get)
    print(f"  Activation ceiling axis: {worst_act} "
          f"(lowest Act@GT = {act_by_axis[worst_act]:.4f})")

# Which axis has biggest oracle CMLt gain?
if oracle_available and "delta_oracle_CMLt" in res_df.columns:
    gain_by_axis = {}
    for ac, an in zip(AXIS_COLS, AXIS_NAMES):
        gain_by_axis[an] = res_df[res_df[ac] == 1].dropna(
            subset=["delta_oracle_CMLt"])["delta_oracle_CMLt"].mean()
    best_gain = max(gain_by_axis, key=gain_by_axis.get)
    print(f"  Tempo ceiling axis:      {best_gain} "
          f"(largest oracle ΔCMLt = {gain_by_axis[best_gain]:+.3f})")

# Which axis is most associated with octave errors?
oct_by_axis = {}
for ac, an in zip(AXIS_COLS, AXIS_NAMES):
    sub = res_df[res_df[ac] == 1]
    oct_by_axis[an] = 100 * (sub["error_cat"] == "octave_error").sum() / len(sub) if len(sub) > 0 else 0
oct_axis = max(oct_by_axis, key=oct_by_axis.get)
print(f"  Octave error axis:       {oct_axis} "
      f"({oct_by_axis[oct_axis]:.1f}% octave errors)")

# Structural/context — clear pattern?
sc = res_df[res_df["axis:Structural/context"] == 1]
sc_n = len(sc)
if sc_n > 0:
    sc_f = sc["F-measure"].mean()
    all_f = res_df["F-measure"].mean()
    sc_dom = max(ERROR_CATS, key=lambda c: (sc["error_cat"] == c).sum() / sc_n - baseline_pcts[c] / 100)
    print(f"  Structural/context:      N={sc_n}, F={sc_f:.3f} (vs {all_f:.3f} overall), "
          f"leans toward '{sc_dom}'")

### 6.5 Activation Bottleneck Quantification

The three interventions in §6.3 all failed to improve F-measure. But do they fail because
the *decoding strategy* is wrong, or because the *activations themselves* cannot support
higher F? We quantify the activation bottleneck in two complementary ways:

1. **Per-track optimal threshold** on real activations — establishes the upper bound of peak-picking.
2. **Ground-truth activations + DBN** (SMC and 4 other datasets) — shows how much higher the DBN could go if the activations were clean.


In [ ]:
# ── Experiment 1: Per-track optimal threshold on real activations ──
import numpy as np
from scipy.special import expit as sigmoid
from scipy.ndimage import maximum_filter1d
import mir_eval
import csv

BT_CACHE = ROOT / "beat_this_activations_cache"
SMC_GT = ROOT / "beat_this_annotations" / "smc" / "annotations" / "beats"
FPS = 50

def pick_beats_at(act, threshold, fps=50):
    """Peak-pick local maxima above threshold (same as beat_this minimal postprocessor)."""
    local_max = maximum_filter1d(act, size=7, mode='constant', cval=0)
    peaks = (act == local_max) & (act > threshold)
    frames = np.where(peaks)[0]
    if len(frames) > 1:
        keep = [frames[0]]
        for f in frames[1:]:
            if f - keep[-1] > 1:
                keep.append(f)
        frames = np.array(keep)
    return frames / fps

def load_gt_beats(path):
    # Robust to both single-column ("time") and multi-column ("time\tbeat_number") .beats formats
    out = []
    for line in open(path):
        line = line.strip()
        if not line: continue
        out.append(float(line.split()[0]))
    return np.array(out)

# Assign error categories (same as Section 5)
def get_category(tid, exp1_df):
    r = exp1_df[exp1_df['track_id'] == tid].iloc[0]
    F = r['btthis_native_F']
    AMLt = r['btthis_native_AMLt']
    CMLt = r['btthis_native_CMLt']
    # CMLc from results.csv
    with open(ROOT / 'results.csv') as f:
        for row in csv.DictReader(f):
            if row['track_id'] == tid:
                CMLc = float(row['CMLc'])
                break
    if F >= 0.8: return 'good'
    if F < 0.3 and AMLt < 0.3: return 'total_failure'
    if AMLt - F > 0.25: return 'octave_error'
    if CMLt - CMLc > 0.2: return 'continuity_drift'
    return 'other'

# Load the combined results CSV
import pandas as pd
# Rename to avoid clobbering the main res_df from section 2
exp1_df = pd.read_csv(ROOT / 'bt_transformer_tempo_constrained_results.csv')

thresholds = np.arange(0.01, 1.0, 0.01)
results_exp1 = {}

for npz_path in sorted(BT_CACHE.glob('*.npz')):
    tid = npz_path.stem
    gt_path = SMC_GT / f'{tid}.beats'
    if not gt_path.exists():
        continue
    ref = load_gt_beats(gt_path)
    act = sigmoid(np.load(npz_path)['beat'])

    best_f, best_thresh, default_f = 0, 0.5, 0
    for t in thresholds:
        est = pick_beats_at(act, t, FPS)
        if len(est) == 0 or len(ref) == 0:
            continue
        f = mir_eval.beat.evaluate(ref, est)['F-measure']
        if f > best_f:
            best_f, best_thresh = f, t
        if abs(t - 0.5) < 0.005:
            default_f = f

    cat = get_category(tid, exp1_df)
    results_exp1[tid] = dict(category=cat, default_F=default_f,
                             best_F=best_f, best_thresh=best_thresh,
                             gain=best_f - default_f)

print('Experiment 1: Per-track optimal threshold on REAL activations')
print('=' * 65)
print(f"{'Category':<20} {'N':>4} {'Default F':>10} {'Best F':>8} {'Gain':>8}")
print('-' * 55)
for cat in ['good', 'octave_error', 'continuity_drift', 'total_failure', 'other']:
    sub = [r for r in results_exp1.values() if r['category'] == cat]
    if not sub: continue
    print(f"  {cat:<18} {len(sub):>4} {np.mean([r['default_F'] for r in sub]):>10.3f}"
          f" {np.mean([r['best_F'] for r in sub]):>8.3f}"
          f" {np.mean([r['gain'] for r in sub]):>+8.3f}")
all_v = list(results_exp1.values())
print(f"  {'ALL':<18} {len(all_v):>4} {np.mean([r['default_F'] for r in all_v]):>10.3f}"
      f" {np.mean([r['best_F'] for r in all_v]):>8.3f}"
      f" {np.mean([r['gain'] for r in all_v]):>+8.3f}")

print(f'\nPer-track detail for total_failure:')
for tid, r in sorted(results_exp1.items()):
    if r['category'] == 'total_failure':
        print(f"  {tid}: default_F={r['default_F']:.3f}, best_F={r['best_F']:.3f}, "
              f"best_thresh={r['best_thresh']:.2f}, gain={r['gain']:+.3f}")

In [ ]:
# ── Experiment 2: Ground-truth activations + DBN (SMC) ──
import torch
from madmom.features.downbeats import DBNDownBeatTrackingProcessor

def build_dbn(min_bpm=30):
    return DBNDownBeatTrackingProcessor(
        beats_per_bar=[3, 4], min_bpm=min_bpm, max_bpm=215.0,
        fps=FPS, transition_lambda=100)

def run_dbn(dbn, beat_logits, downbeat_logits):
    """Run DBN on logits, return beat times."""
    beat_prob = torch.from_numpy(beat_logits).double().sigmoid()
    downbeat_prob = torch.from_numpy(downbeat_logits).double().sigmoid()
    eps = 1e-5
    beat_prob = beat_prob * (1 - eps) + eps / 2
    downbeat_prob = downbeat_prob * (1 - eps) + eps / 2
    combined = np.vstack((
        np.maximum(beat_prob.numpy() - downbeat_prob.numpy(), eps / 2),
        downbeat_prob.numpy(),
    )).T
    results = [hmm.viterbi(combined) for hmm in dbn.hmms]
    best_idx = np.argmax([r[1] for r in results])
    path, _ = results[best_idx]
    om = dbn.hmms[best_idx].observation_model
    beats = np.empty(0, dtype=int)
    beat_range = om.pointers[path] >= 1
    if beat_range.any():
        idx = np.nonzero(np.diff(beat_range.astype(int)))[0] + 1
        if beat_range[0]: idx = np.r_[0, idx]
        if beat_range[-1]: idx = np.r_[idx, beat_range.size]
        if idx.any():
            for left, right in idx.reshape((-1, 2)):
                peak = np.argmax(combined[left:right]) // 2 + left
                beats = np.hstack((beats, peak))
    return beats / float(FPS) if len(beats) > 0 else np.array([])

def make_gt_logits(ref, n_frames, fps=50, sigma=2):
    """Create synthetic logits with Gaussian peaks at GT beat positions."""
    beat_logits = np.full(n_frames, -5.0, dtype=np.float32)
    db_logits = np.full(n_frames, -5.0, dtype=np.float32)
    for i, bt in enumerate(ref):
        c = int(bt * fps)
        lo, hi = max(0, c - 5*sigma), min(n_frames, c + 5*sigma + 1)
        for f in range(lo, hi):
            val = 5.0 * np.exp(-0.5 * ((f - c) / sigma) ** 2)
            beat_logits[f] = max(beat_logits[f], val)
            if i % 4 == 0:
                db_logits[f] = max(db_logits[f], val)
    return beat_logits, db_logits

import warnings
warnings.filterwarnings('ignore')

dbn_55 = build_dbn(min_bpm=55)
dbn_30 = build_dbn(min_bpm=30)

# Run on SMC (from cache). Record full mir_eval metric dict for GT+DBN min30
# so Table 3 can cite both F, CMLt and AMLt.
real_peak, real_55, real_30, gt_55 = [], [], [], []
gt_dbn30_rows = []   # list of dicts: track_id, F, CMLt, AMLt, Cemgil
for npz_path in sorted(BT_CACHE.glob('*.npz')):
    tid = npz_path.stem
    gt_path = SMC_GT / f'{tid}.beats'
    if not gt_path.exists(): continue
    ref = load_gt_beats(gt_path)
    if len(ref) < 2: continue
    data = np.load(npz_path)
    bn, dn = data['beat'], data['downbeat']
    n_frames = len(bn)
    act = sigmoid(bn)

    est = pick_beats_at(act, 0.5, FPS)
    s = mir_eval.beat.evaluate(ref, est) if len(est) > 0 else {'F-measure': 0}
    real_peak.append(s['F-measure'])

    gt_b, gt_d = make_gt_logits(ref, n_frames, FPS)
    for dbn, rl, gl, record_full in [(dbn_55, real_55, gt_55, False),
                                      (dbn_30, real_30, None,   True)]:
        e1 = run_dbn(dbn, bn, dn)
        s1 = mir_eval.beat.evaluate(ref, e1) if len(e1) > 0 else {'F-measure': 0, 'Correct Metric Level Total': 0, 'Any Metric Level Total': 0}
        rl.append(s1['F-measure'])
        e2 = run_dbn(dbn, gt_b, gt_d)
        s2 = mir_eval.beat.evaluate(ref, e2) if len(e2) > 0 else {'F-measure': 0, 'Correct Metric Level Total': 0, 'Any Metric Level Total': 0}
        if record_full:
            gt_dbn30_rows.append({
                'track_id': tid,
                'F':    s2['F-measure'],
                'CMLt': s2['Correct Metric Level Total'],
                'AMLt': s2['Any Metric Level Total'],
            })
        else:
            gl.append(s2['F-measure'])

# Build gt_30 F list from the richer record
gt_30 = [r['F'] for r in gt_dbn30_rows]

# Persist per-track GT+DBN (min30) results for Table 3 and downstream reporting
import pandas as pd
_gt_act_df = pd.DataFrame(gt_dbn30_rows)
_gt_act_df.to_csv(ROOT / "gt_activation_dbn_per_track.csv", index=False)

print('Experiment 2: Ground-truth activations + DBN (SMC)')
print('=' * 65)
print(f"  {'Condition':<30} {'Mean F':>8}  {'Mean CMLt':>10}  {'Mean AMLt':>10}  {'F<0.5':>5}")
print(f"  {'-'*65}")
print(f"  {'Real act + peak-pick':<30} {np.mean(real_peak):>8.3f}")
print(f"  {'Real act + DBN (min55)':<30} {np.mean(real_55):>8.3f}  {'':>10}  {'':>10}  {sum(1 for f in real_55 if f<0.5):>5}")
print(f"  {'Real act + DBN (min30)':<30} {np.mean(real_30):>8.3f}  {'':>10}  {'':>10}  {sum(1 for f in real_30 if f<0.5):>5}")
print(f"  {'GT act + DBN (min55)':<30} {np.mean(gt_55):>8.3f}  {'':>10}  {'':>10}  {sum(1 for f in gt_55 if f<0.5):>5}")
print(f"  {'GT act + DBN (min30)':<30} {_gt_act_df['F'].mean():>8.3f}  "
      f"{_gt_act_df['CMLt'].mean():>10.3f}  {_gt_act_df['AMLt'].mean():>10.3f}  "
      f"{int((_gt_act_df['F']<0.5).sum()):>5}")
print(f"\n  Gap (GT+DBN30 - Real+DBN30): {_gt_act_df['F'].mean()-np.mean(real_30):+.3f}")


In [ ]:
# ── Experiment 2 (cross-dataset): GT activations + DBN on other datasets ──
# Requires: audio files in ~/mnt/db_1/jaehoon/beat-tracking/labeled_data/
#           beat_this fold0 checkpoint

from beat_this.inference import Audio2Frames
from beat_this.preprocessing import load_audio

BASE_DATA = Path(os.path.expanduser('~/mnt/db_1/jaehoon/beat-tracking/labeled_data'))

def simple_mapper(label_path, audio_dir):
    return audio_dir / (label_path.stem + '.wav')

def gtzan_mapper(label_path, audio_dir):
    key = label_path.stem.replace('gtzan_', '')
    genre, num = key.rsplit('_', 1)
    audio_key = f'{genre}.{num}'
    for f in audio_dir.glob(f'*_{audio_key}.wav'):
        return f
    return None

datasets = [
    ('ballroom', BASE_DATA/'ballroom'/'label', BASE_DATA/'ballroom'/'data', '*.beats', simple_mapper),
    ('beatles',  BASE_DATA/'beatles'/'label',  BASE_DATA/'beatles'/'data',  '*.txt',   simple_mapper),
    ('gtzan',    BASE_DATA/'gtzan'/'label',    BASE_DATA/'gtzan'/'data',    '*.beats', gtzan_mapper),
    ('hains',    BASE_DATA/'hains'/'label',    BASE_DATA/'hains'/'data',    '*.txt',   simple_mapper),
]

device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
print(f'Loading beat_this fold0 on {device}...')
a2f = Audio2Frames(checkpoint_path='fold0', device=device)

all_results = {}

for ds_name, label_dir, audio_dir, label_glob, audio_mapper in datasets:
    print(f'\n{"="*70}')
    print(f'DATASET: {ds_name}')
    print(f'{"="*70}')

    real_peak, real_55, real_30, gt_55, gt_30 = [], [], [], [], []
    n_processed, n_skipped = 0, 0

    for label_path in sorted(label_dir.glob(label_glob)):
        audio_path = audio_mapper(label_path, audio_dir)
        if audio_path is None or not audio_path.exists():
            n_skipped += 1; continue
        ref = load_gt_beats(label_path)
        if len(ref) < 2:
            n_skipped += 1; continue
        try:
            signal, sr = load_audio(str(audio_path))
            with torch.no_grad():
                bl, dl = a2f(signal, sr)
            bn, dn = bl.cpu().numpy(), dl.cpu().numpy()
        except:
            n_skipped += 1; continue

        n_frames = len(bn)
        act = sigmoid(bn)
        est = pick_beats_at(act, 0.5, FPS)
        s = mir_eval.beat.evaluate(ref, est) if len(est) > 0 else {'F-measure': 0}
        real_peak.append(s['F-measure'])

        gt_b, gt_d = make_gt_logits(ref, n_frames, FPS)
        for dbn, rl, gl in [(dbn_55, real_55, gt_55), (dbn_30, real_30, gt_30)]:
            e1 = run_dbn(dbn, bn, dn)
            s1 = mir_eval.beat.evaluate(ref, e1) if len(e1) > 0 else {'F-measure': 0}
            rl.append(s1['F-measure'])
            e2 = run_dbn(dbn, gt_b, gt_d)
            s2 = mir_eval.beat.evaluate(ref, e2) if len(e2) > 0 else {'F-measure': 0}
            gl.append(s2['F-measure'])

        n_processed += 1
        if n_processed % 100 == 0:
            print(f'  processed {n_processed}...')

    all_results[ds_name] = dict(
        n=n_processed, real_peak=np.mean(real_peak),
        real_55=np.mean(real_55), real_30=np.mean(real_30),
        gt_55=np.mean(gt_55), gt_30=np.mean(gt_30),
        gt55_low=sum(1 for f in gt_55 if f<0.5),
        gt30_low=sum(1 for f in gt_30 if f<0.5))

    print(f'  Processed: {n_processed}, Skipped: {n_skipped}')
    print(f"\n  {'Condition':<30} {'Mean F':>8}")
    print(f"  {'-'*40}")
    print(f"  {'Real act + peak-pick':<30} {np.mean(real_peak):>8.3f}")
    print(f"  {'Real act + DBN (min55)':<30} {np.mean(real_55):>8.3f}")
    print(f"  {'Real act + DBN (min30)':<30} {np.mean(real_30):>8.3f}")
    print(f"  {'GT act + DBN (min55)':<30} {np.mean(gt_55):>8.3f}")
    print(f"  {'GT act + DBN (min30)':<30} {np.mean(gt_30):>8.3f}")
    print(f"\n  Tracks F<0.5: GT55={sum(1 for f in gt_55 if f<0.5)}, GT30={sum(1 for f in gt_30 if f<0.5)}")

# Summary table
print(f'\n{"="*80}')
print('CROSS-DATASET SUMMARY')
print(f'{"="*80}')
print(f"{'Dataset':<12} {'N':>5} {'Real+peak':>10} {'Real+DBN55':>11} {'Real+DBN30':>11} {'GT+DBN55':>9} {'GT+DBN30':>9} {'Gap':>7}")
print('-' * 80)
for ds in ['ballroom', 'beatles', 'gtzan', 'hains']:
    r = all_results[ds]
    gap = r['gt_30'] - r['real_30']
    print(f"{ds:<12} {r['n']:>5} {r['real_peak']:>10.3f} {r['real_55']:>11.3f} {r['real_30']:>11.3f} {r['gt_55']:>9.3f} {r['gt_30']:>9.3f} {gap:>+7.3f}")
# Add SMC from experiment 2
smc_gap = np.mean(gt_30) - np.mean(real_30)  # from previous cell's SMC results
print(f"{'SMC':<12} {217:>5} {0.627:>10.3f} {0.576:>11.3f} {0.585:>11.3f} {0.827:>9.3f} {0.924:>9.3f} {+0.339:>+7.3f}")
print(f'\nThe activation bottleneck scales with dataset difficulty.')
print(f'SMC gap (+0.339) is 4x larger than any other dataset.')

## 7. The Role of Tempo

The activation ceiling established in §6 explains much of the F-measure gap but says nothing about metrical coherence (CMLt/AMLt). We now turn to the second dimension of beat tracking: tempo. The questions we address:

1. When the DBN post-processes activations, does it help or hurt? (§7.1)
2. Does the DBN's default `min_bpm=55` cutoff silently force 21% of SMC tracks into double-tempo predictions? (§7.2)
3. How accurate are Beat Transformer's and madmom TCN's tempo heads on SMC? (§7.3)
4. If we inject the correct tempo into the DBN, does F-measure recover? (§7.4-§7.7)
5. Is the DBN's `transition_lambda` parameter miscalibrated for SMC? Can a per-track oracle close the gap? (§7.8-§7.9)


### 7.1 DBN Postprocessing: The Tempo-Continuity Tradeoff

We analyze the effect of adding madmom's DBN post-processing to beat_this activations. The DBN imposes a rigid tempo-continuity prior — we show it improves metrical coherence (AMLt) but hurts placement precision (F) on SMC, especially on tempo-instability tracks.


In [21]:
# -- Aggregate DBN impact --
res_df["delta_F_dbn"] = res_df["dbn_F-measure"] - res_df["F-measure"]
res_df["delta_AMLt_dbn"] = res_df["dbn_AMLt"] - res_df["AMLt"]

n_worse = (res_df["delta_F_dbn"] < -0.001).sum()
n_better = (res_df["delta_F_dbn"] > 0.001).sum()
n_same = len(res_df) - n_worse - n_better

print("DBN Postprocessing Impact (beat_this activations)")
print("=" * 60)
print(f"  {'Metric':<20} {'Raw':>8} {'+ DBN':>8} {'Delta':>8}")
print("-" * 50)
for m, dm in [("F-measure", "dbn_F-measure"), ("CMLt", "dbn_CMLt"),
              ("AMLt", "dbn_AMLt"), ("Goto", "dbn_Goto")]:
    raw = res_df[m].mean()
    dbn = res_df[dm].mean()
    print(f"  {m:<20} {raw:>8.3f} {dbn:>8.3f} {dbn-raw:>+8.3f}")

print(f"\n  Tracks worsened: {n_worse},  improved: {n_better},  unchanged: {n_same}")
print(f"  Goto > 0.5:  raw={int((res_df['Goto']>0.5).sum())} -> DBN={int((res_df['dbn_Goto']>0.5).sum())}")

# Per-track oracle
oracle_f = res_df[["F-measure", "dbn_F-measure"]].max(axis=1).mean()
print(f"\n  Per-track oracle (best of raw/DBN): F={oracle_f:.3f}")

DBN Postprocessing Impact (beat_this activations)
  Metric                    Raw    + DBN    Delta
--------------------------------------------------
  F-measure               0.627    0.576   -0.051
  CMLt                    0.514    0.474   -0.040
  AMLt                    0.610    0.656   +0.046
  Goto                    0.166    0.267   +0.101

  Tracks worsened: 126,  improved: 62,  unchanged: 29
  Goto > 0.5:  raw=36 -> DBN=58

  Per-track oracle (best of raw/DBN): F=0.647


In [22]:
# -- DBN impact by tag --
print("DBN Impact by Tag (sorted by delta F)")
print("=" * 60)
print(f"  {'Tag':<40} {'N':>4} {'dF':>8}")
print("-" * 55)
tag_dbn = []
for tag in tag_cols:
    name = tag.replace("tag:", "")
    sub = res_df[res_df[tag] == 1]
    if len(sub) >= 3:
        tag_dbn.append((name, len(sub), sub["delta_F_dbn"].mean()))
tag_dbn.sort(key=lambda x: x[2])
for name, n, delta in tag_dbn:
    print(f"  {name:<40} {n:>4} {delta:>+8.3f}")

# DBN impact by confidence
print(f"\nDBN Impact by Confidence:")
for conf in sorted(res_df["confidence"].unique()):
    sub = res_df[res_df["confidence"] == conf]
    print(f"  Confidence {conf}: dF = {sub['delta_F_dbn'].mean():+.3f} (n={len(sub)})")

# DBN impact by error category
print(f"\nDBN Impact by Error Category:")
for cat in ["good", "octave_error", "continuity_drift", "total_failure", "other"]:
    sub = res_df[res_df["error_cat"] == cat]
    if len(sub) > 0:
        print(f"  {cat:<20}: dF = {sub['delta_F_dbn'].mean():+.3f}, "
              f"dAMLt = {sub['delta_AMLt_dbn'].mean():+.3f} (n={len(sub)})")

DBN Impact by Tag (sorted by delta F)
  Tag                                         N       dF
-------------------------------------------------------
  low familiarity                            23   -0.121
  changing time signature                    14   -0.084
  ternary meter                              70   -0.081
  lack of harmonic change                     7   -0.074
  rich ornamentation in the interpretation   25   -0.073
  expressive timing                         124   -0.068
  missing bass                               72   -0.067
  wide dynamic range                         13   -0.067
  pause                                      33   -0.061
  no repetition                               6   -0.058
  tempo discontinuity                        42   -0.056
  slow tempo                                 71   -0.055
  lack of transient sounds                   70   -0.047
  strong syncopation                         24   -0.041
  poor sound quality                         21   -

In [23]:
# -- DBN confidence (Viterbi log-prob) as selector signal --
from scipy.stats import spearmanr

# Log-prob is already in results.csv
rho_abs, p_abs = spearmanr(res_df["dbn_norm_log_prob"], res_df["dbn_F-measure"])
rho_delta, p_delta = spearmanr(res_df["dbn_norm_log_prob"], res_df["delta_F_dbn"])

print("DBN Confidence (Viterbi Log-Probability) as Selector")
print("=" * 60)
print(f"  Correlation with dbn_F:  rho={rho_abs:+.3f} (p={p_abs:.2e})")
print(f"  Correlation with delta_F: rho={rho_delta:+.3f} (p={p_delta:.2e})")

# Sweep thresholds for selector: use DBN when log_prob > threshold, else raw
print(f"\n  Selector sweep (use DBN only when norm_log_prob > threshold):")
print(f"  {'Threshold':>10} {'N(DBN)':>7} {'Mean F':>8}")
print("-" * 30)
best_f = 0
best_t = -4.0
for t in np.arange(-4.0, -1.0, 0.25):
    use_dbn = res_df["dbn_norm_log_prob"] > t
    selected_f = np.where(use_dbn, res_df["dbn_F-measure"], res_df["F-measure"])
    mean_f = selected_f.mean()
    if mean_f > best_f:
        best_f = mean_f
        best_t = t
    print(f"  {t:>10.2f} {use_dbn.sum():>7} {mean_f:>8.4f}")

print(f"\n  Always raw: F={res_df['F-measure'].mean():.4f}")
print(f"  Best selector: F={best_f:.4f} at threshold={best_t:.2f}")
print(f"  -> DBN confidence is too weak as a per-track selector; distributions overlap.")

DBN Confidence (Viterbi Log-Probability) as Selector
  Correlation with dbn_F:  rho=+0.549 (p=1.84e-18)
  Correlation with delta_F: rho=+0.418 (p=1.38e-10)

  Selector sweep (use DBN only when norm_log_prob > threshold):
   Threshold  N(DBN)   Mean F
------------------------------
       -4.00     216   0.5780
       -3.75     216   0.5780
       -3.50     216   0.5780
       -3.25     215   0.5785
       -3.00     206   0.5839
       -2.75      64   0.6245
       -2.50       0   0.6270
       -2.25       0   0.6270
       -2.00       0   0.6270
       -1.75       0   0.6270
       -1.50       0   0.6270
       -1.25       0   0.6270

  Always raw: F=0.6270
  Best selector: F=0.6270 at threshold=-2.50
  -> DBN confidence is too weak as a per-track selector; distributions overlap.


### 7.2 Widening the DBN Tempo Range

The paper uses `DBNBeatTrackingProcessor(min_bpm=55, max_bpm=215)`. But 21% of SMC tracks have GT tempo below 55 BPM, forcing the DBN to pick double tempo. Simply widening to `min_bpm=30` yields free gains.


In [27]:
# Sweep min_bpm for Beat Transformer's own DBN
dbn_configs = [
    ("DBN [55, 215] (paper)", 55, 215),
    ("DBN [50, 215]", 50, 215),
    ("DBN [45, 215]", 45, 215),
    ("DBN [40, 215]", 40, 215),
    ("DBN [35, 215]", 35, 215),
    ("DBN [30, 215] (wide)", 30, 215),
]

sweep_rows = []
for name, mn, mx in dbn_configs:
    dbn = DBNBeatTrackingProcessor(
        min_bpm=mn, max_bpm=mx, fps=BT_FPS,
        transition_lambda=100, observation_lambda=6, num_tempi=None, threshold=0.2
    )
    mireval_f = []; mireval_c = []; mireval_a = []
    madmom_f = []; madmom_c = []; madmom_a = []

    for tid in track_ids:
        ba = bt_results[tid]["beat_act"]
        ref_raw = load_gt(tid)
        ref_q = quantize_gt(ref_raw, len(ba))
        pred = dbn(ba)

        r = eval_mireval(ref_raw, pred)
        mireval_f.append(r["F"]); mireval_c.append(r["CMLt"]); mireval_a.append(r["AMLt"])
        r2 = eval_madmom(pred, ref_q, klapuri=True)
        madmom_f.append(r2["F"]); madmom_c.append(r2["CMLt"]); madmom_a.append(r2["AMLt"])

    sweep_rows.append({
        "config": name,
        "mir_F": np.mean(mireval_f), "mir_CMLt": np.mean(mireval_c), "mir_AMLt": np.mean(mireval_a),
        "mm_F": np.mean(madmom_f), "mm_CMLt": np.mean(madmom_c), "mm_AMLt": np.mean(madmom_a),
    })

print("Beat Transformer activations + DBN sweep (Beat Transformer's own beats)")
print(f"\n{'Config':<25} {'mir_F':>8} {'mir_CMLt':>10} {'mir_AMLt':>10} | {'mm_F':>8} {'mm_CMLt':>10} {'mm_AMLt':>10}")
print("-" * 95)
for r in sweep_rows:
    print(f"{r['config']:<25} {r['mir_F']:>8.4f} {r['mir_CMLt']:>10.4f} {r['mir_AMLt']:>10.4f} | "
          f"{r['mm_F']:>8.4f} {r['mm_CMLt']:>10.4f} {r['mm_AMLt']:>10.4f}")
print(f"\n{'Paper reports':<25} {'':>8} {'':>10} {'':>10} | {'0.596':>8} {'0.521':>10} {'0.692':>10}")

Beat Transformer activations + DBN sweep (Beat Transformer's own beats)

Config                       mir_F   mir_CMLt   mir_AMLt |     mm_F    mm_CMLt    mm_AMLt
-----------------------------------------------------------------------------------------------
DBN [55, 215] (paper)       0.5656     0.4639     0.6316 |   0.5955     0.5077     0.6974
DBN [50, 215]               0.5704     0.4829     0.6339 |   0.6017     0.5266     0.6994
DBN [45, 215]               0.5776     0.5075     0.6354 |   0.6090     0.5500     0.7009
DBN [40, 215]               0.5775     0.5078     0.6394 |   0.6090     0.5499     0.7044
DBN [35, 215]               0.5791     0.5171     0.6374 |   0.6105     0.5587     0.7026
DBN [30, 215] (wide)        0.5805     0.5232     0.6433 |   0.6115     0.5636     0.7058

Paper reports                                            |    0.596      0.521      0.692


### 7.3 Tempo Estimation Accuracy

Beat Transformer has a 300-way tempo classification head (argmax gives integer BPM). We compare its accuracy against ground-truth tempo derived from median IBI, and compare against madmom TCN's standalone tempo head.


In [28]:
# Tempo accuracy analysis
tempo_rows = []
for tid in track_ids:
    ref = load_gt(tid)
    gt_bpm = gt_tempo(ref)
    bt_bpm = bt_results[tid]["tempo_bpm"]
    ratio = bt_bpm / gt_bpm if gt_bpm > 0 else 0
    tempo_rows.append({
        "track_id": tid, "gt_bpm": gt_bpm, "bt_bpm": bt_bpm,
        "ratio": ratio, "class": classify_tempo(ratio)
    })

tempo_df = pd.DataFrame(tempo_rows)
counts = tempo_df["class"].value_counts()
total = len(tempo_df)

print("Beat Transformer Tempo Head Accuracy on SMC")
print("=" * 50)
for cls in ["correct", "double", "half", "other"]:
    n = counts.get(cls, 0)
    print(f"  {cls:<10} {n:>4}  ({100*n/total:>5.1f}%)")
print(f"\n  (Reference: madmom TCN was correct on 22% of SMC)")

# GT tempo distribution
print(f"\nGT tempo distribution:")
print(f"  min={tempo_df['gt_bpm'].min():.1f}  median={tempo_df['gt_bpm'].median():.1f}  "
      f"max={tempo_df['gt_bpm'].max():.1f}")
print(f"  Tracks with GT tempo < 55 BPM: {(tempo_df['gt_bpm'] < 55).sum()}/217 ({100*(tempo_df['gt_bpm'] < 55).mean():.1f}%)")
print(f"  Tracks with GT tempo < 40 BPM: {(tempo_df['gt_bpm'] < 40).sum()}/217")

Beat Transformer Tempo Head Accuracy on SMC
  correct     127  ( 58.5%)
  double       21  (  9.7%)
  half         22  ( 10.1%)
  other        47  ( 21.7%)

  (Reference: madmom TCN was correct on 22% of SMC)

GT tempo distribution:
  min=32.7  median=70.9  max=206.9
  Tracks with GT tempo < 55 BPM: 45/217 (20.7%)
  Tracks with GT tempo < 40 BPM: 13/217


In [ ]:
# -- Figure 4: Beat Transformer tempo head accuracy by BPM range --
import matplotlib.pyplot as plt
import numpy as np

ranges = [(0, 55), (55, 70), (70, 90), (90, 120), (120, 9999)]
labels = ["<55", "55-70", "70-90", "90-120", ">120"]
counts = {"correct": [], "double": [], "half": [], "other": []}
ns = []
for lo, hi in ranges:
    sub = [r for r in tempo_rows if lo <= r["gt_bpm"] < hi]
    ns.append(len(sub))
    for cls in counts:
        counts[cls].append(sum(1 for r in sub if r["class"] == cls))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(labels))
w = 0.65
bottoms = np.zeros(len(labels))
colors  = {"correct": "#2ca02c", "double": "#f6b26b", "half": "#c67e7e", "other": "#a0a0a0"}
lab_map = {"correct": "Correct", "double": "Double tempo (faster)",
           "half": "Half tempo (slower)", "other": "Other wrong"}
for cls in ["correct", "double", "half", "other"]:
    ax.bar(x, counts[cls], w, bottom=bottoms, label=lab_map[cls],
           color=colors[cls], edgecolor='black', linewidth=0.3)
    bottoms += np.array(counts[cls])

for i in range(len(ranges)):
    if ns[i] > 0:
        pct = 100 * counts["correct"][i] / ns[i]
        ax.text(i, bottoms[i] + 1, f"{pct:.0f}%", ha='center',
                fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(labels, ns)])
ax.set_xlabel("Ground truth BPM range")
ax.set_ylabel("Number of tracks")
ax.set_title("Beat Transformer tempo head accuracy on SMC")
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

total_correct = sum(counts["correct"])
print(f"Overall tempo head accuracy: {total_correct}/217 ({100*total_correct/217:.0f}%)")


### 7.4 Tempo-Constrained DBN (Beat Transformer)

We test whether Beat Transformer's own tempo prediction can improve its beat tracking through DBN constraint. Three strategies: (1) Top-1 narrow, (2) Top-6 union wide, (3) Viterbi-select.


In [29]:
def dbn_bt(ba, mn, mx):
    """Run Beat-Transformer-fps DBN with given BPM range."""
    mn = max(30.0, mn); mx = min(300.0, mx)
    if mx <= mn + 5: mx = mn + 5
    d = DBNBeatTrackingProcessor(min_bpm=mn, max_bpm=mx, fps=BT_FPS,
                                  transition_lambda=100, observation_lambda=6,
                                  num_tempi=None, threshold=0.2)
    return d(ba)

def nms_peaks(tempo_logits, nms_dist=8, min_bpm=20):
    """NMS-filtered peaks from tempo logits, with softmax probabilities."""
    e = np.exp(tempo_logits - tempo_logits.max())
    p = e / e.sum()
    order = np.argsort(-tempo_logits)
    picked = []
    for i in order:
        if i < min_bpm: continue
        if all(abs(i - j) >= nms_dist for j in picked):
            picked.append(int(i))
    return [(b, float(p[b])) for b in picked]

def viterbi_select(ba, tempo_logits, K=5, pct=0.20):
    """Wide + top-K narrow candidates, pick by Viterbi log-prob."""
    def decode(mn, mx):
        mn = max(30.0, mn); mx = min(300.0, mx)
        if mx <= mn + 5: mx = mn + 5
        d = DBNBeatTrackingProcessor(min_bpm=mn, max_bpm=mx, fps=BT_FPS,
                                      transition_lambda=100, observation_lambda=6,
                                      num_tempi=None, threshold=0.2)
        act, first = threshold_activations(ba, 0.2)
        if not act.any(): return np.empty(0), -np.inf
        path, lp = d.hmm.viterbi(act)
        if not path.any(): return np.empty(0), -np.inf
        br = d.om.pointers[path]
        idx = np.nonzero(np.diff(br))[0] + 1
        if br[0]: idx = np.r_[0, idx]
        if br[-1]: idx = np.r_[idx, br.size]
        beats = [np.argmax(act[l:r]) + l for l, r in idx.reshape((-1, 2))]
        return (np.array(beats) + first) / BT_FPS, lp

    peaks = nms_peaks(tempo_logits)[:K]
    cands = [decode(30, 215)]
    for b, _ in peaks:
        cands.append(decode(b * (1 - pct), b * (1 + pct)))
    return max(cands, key=lambda x: x[1])[0]

# Run all Beat-Transformer-on-itself strategies
bt_on_bt_rows = []
for tid in tqdm(track_ids, desc="Beat-Transformer-on-itself"):
    ba = bt_results[tid]["beat_act"]
    tl = bt_results[tid]["tempo_logits"]
    bt_bpm = bt_results[tid]["tempo_bpm"]
    ref = load_gt(tid)
    gbpm = gt_tempo(ref)

    # Paper default [55, 215]
    beats_paper = dbn_bt(ba, 55, 215)
    # Wide [30, 215]
    beats_wide = dbn_bt(ba, 30, 215)
    # Top-1 ±20%
    beats_top1 = dbn_bt(ba, bt_bpm * 0.8, bt_bpm * 1.2) if bt_bpm > 0 else beats_wide
    # Top-6 union
    peaks = nms_peaks(tl)[:6]
    if peaks:
        beats_topk = dbn_bt(ba, min(b for b, _ in peaks) * 0.8, max(b for b, _ in peaks) * 1.2)
    else:
        beats_topk = beats_wide
    # Viterbi-select
    beats_viterbi = viterbi_select(ba, tl)
    # Oracle
    beats_oracle = dbn_bt(ba, gbpm * 0.8, gbpm * 1.2) if gbpm > 0 else beats_wide

    row = {"track_id": tid, "gt_bpm": gbpm, "bt_bpm": bt_bpm}
    for label, beats in [("paper", beats_paper), ("wide", beats_wide), ("top1", beats_top1),
                          ("topk", beats_topk), ("viterbi", beats_viterbi), ("oracle", beats_oracle)]:
        r = eval_mireval(ref, beats)
        row[f"{label}_F"] = r["F"]; row[f"{label}_CMLt"] = r["CMLt"]; row[f"{label}_AMLt"] = r["AMLt"]
    bt_on_bt_rows.append(row)

# Summary
print(f"\n{'System':<30} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
print("=" * 58)
for label, name in [("paper", "Beat Transformer + DBN [55,215] (paper)"), ("wide", "Beat Transformer + DBN [30,215] (wide)"),
                     ("top1", "Beat Transformer + tempo top-1 ±20%"), ("topk", "Beat Transformer + tempo top-6 union"),
                     ("viterbi", "Beat Transformer + Viterbi-select"), ("oracle", "Beat Transformer + oracle (GT tempo)")]:
    f = np.mean([r[f"{label}_F"] for r in bt_on_bt_rows])
    c = np.mean([r[f"{label}_CMLt"] for r in bt_on_bt_rows])
    a = np.mean([r[f"{label}_AMLt"] for r in bt_on_bt_rows])
    print(f"  {name:<28} {f:>8.4f} {c:>8.4f} {a:>8.4f}")

Beat-Transformer-on-itself:   0%|          | 0/217 [00:00<?, ?it/s]

Beat-Transformer-on-itself:   0%|          | 1/217 [00:00<00:23,  9.24it/s]

Beat-Transformer-on-itself:   1%|▏         | 3/217 [00:00<00:21,  9.74it/s]

Beat-Transformer-on-itself:   2%|▏         | 4/217 [00:00<00:22,  9.33it/s]

Beat-Transformer-on-itself:   2%|▏         | 5/217 [00:00<00:23,  8.99it/s]

...

Beat-Transformer-on-itself:  98%|█████████▊| 213/217 [00:23<00:00,  9.81it/s]

Beat-Transformer-on-itself:  99%|█████████▉| 215/217 [00:23<00:00, 10.05it/s]

Beat-Transformer-on-itself: 100%|██████████| 217/217 [00:23<00:00, 10.06it/s]

Beat-Transformer-on-itself: 100%|██████████| 217/217 [00:23<00:00,  9.06it/s]


System                                F     CMLt     AMLt
  Beat Transformer + DBN [55,215] (paper)   0.5656   0.4639   0.6316
  Beat Transformer + DBN [30,215] (wide)   0.5805   0.5232   0.6433
  Beat Transformer + tempo top-1 ±20%   0.5670   0.4927   0.6348
  Beat Transformer + tempo top-6 union   0.5802   0.5224   0.6433
  Beat Transformer + Viterbi-select   0.5818   0.5277   0.6458
  Beat Transformer + oracle (GT tempo)   0.6267   0.7123   0.7263


### 7.5 Per-Tempo-Class Breakdown

How does the tempo constraint affect tracks where the tempo prediction is correct vs. wrong?


In [30]:
# Merge tempo class into bt_on_bt_rows
tid_to_class = {r["track_id"]: r["class"] for r in tempo_rows}
for r in bt_on_bt_rows:
    r["tempo_class"] = tid_to_class[r["track_id"]]

print(f"{'Tempo class':<12} {'N':>4} | {'paper_F':>8} {'wide_F':>8} {'top1_F':>8} {'viterbi_F':>10} {'oracle_F':>10}")
print("-" * 70)
for cls in ["correct", "double", "half", "other"]:
    sub = [r for r in bt_on_bt_rows if r["tempo_class"] == cls]
    if not sub: continue
    print(f"  {cls:<10} {len(sub):>4} | "
          f"{np.mean([r['paper_F'] for r in sub]):>8.4f} "
          f"{np.mean([r['wide_F'] for r in sub]):>8.4f} "
          f"{np.mean([r['top1_F'] for r in sub]):>8.4f} "
          f"{np.mean([r['viterbi_F'] for r in sub]):>10.4f} "
          f"{np.mean([r['oracle_F'] for r in sub]):>10.4f}")

Tempo class     N |  paper_F   wide_F   top1_F  viterbi_F   oracle_F
----------------------------------------------------------------------
  correct     127 |   0.6610   0.6777   0.6912     0.6782     0.6913
  double       21 |   0.4154   0.4340   0.4430     0.4340     0.5482
  half         22 |   0.5516   0.5456   0.4676     0.5546     0.6677
  other        47 |   0.3815   0.3993   0.3333     0.4002     0.4680


### 7.6 GT-Tempo Upper Bound (Cross-Model)

Now we test whether Beat Transformer's tempo predictions can improve **beat_this**'s beat tracking (a different, stronger acoustic model). This also gives us the GT-tempo upper bound on metrical coherence.


In [31]:
if not HAVE_BEAT_THIS:
    print("Skipping cross-model section (beat_this data not available)")
else:
    def dbn_btthis(beat_logits, db_logits, mn, mx):
        """Run beat_this-fps DBNDownBeatTrackingProcessor."""
        mn = max(30.0, mn); mx = min(300.0, mx)
        if mx <= mn + 5: mx = mn + 5
        d = DBNDownBeatTrackingProcessor(beats_per_bar=[3, 4], min_bpm=mn, max_bpm=mx,
                                          fps=BTTHIS_FPS, transition_lambda=100)
        bp = beat_logits.double().sigmoid().numpy()
        dp = db_logits.double().sigmoid().numpy()
        eps = 1e-5
        bp = bp * (1 - eps) + eps / 2
        dp = dp * (1 - eps) + eps / 2
        combined = np.vstack((np.maximum(bp - dp, eps / 2), dp)).T
        return d(combined)[:, 0]
    
    # Load beat_this predictions and activations
    bt_raw_dir = ROOT / "beat_this_output"
    bt_dbn_dir = ROOT / "beat_this_output_dbn"
    
    cross_rows = []
    for tid in tqdm(track_ids, desc="Cross-model"):
        ref = load_gt(tid)
        gbpm = gt_tempo(ref)
        bt_bpm = bt_results[tid]["tempo_bpm"]
        tl = bt_results[tid]["tempo_logits"]
    
        # Load beat_this activations
        bt_cache = np.load(BT_ACT_CACHE / f"{tid}.npz")
        bl = torch.from_numpy(bt_cache["beat"])
        dl = torch.from_numpy(bt_cache["downbeat"])
    
        # beat_this native (no DBN)
        raw_path = bt_raw_dir / f"{tid.upper()}.beats"
        bt_raw = np.array([float(l.split("\t")[0]) for l in open(raw_path) if l.strip()]) if raw_path.exists() else np.array([])
        # beat_this + default DBN [55, 215]
        dbn_path = bt_dbn_dir / f"{tid.upper()}.beats"
        bt_dbn = np.array([float(l.split("\t")[0]) for l in open(dbn_path) if l.strip()]) if dbn_path.exists() else np.array([])
    
        # beat_this + wide DBN [30, 215]
        bt_wide = dbn_btthis(bl, dl, 30, 215)
        # beat_this + Beat Transformer tempo top-1 ±20%
        bt_top1 = dbn_btthis(bl, dl, bt_bpm * 0.8, bt_bpm * 1.2) if bt_bpm > 0 else bt_dbn
        # beat_this + Beat Transformer tempo top-6 union
        peaks = nms_peaks(tl)[:6]
        if peaks:
            bt_topk = dbn_btthis(bl, dl, min(b for b, _ in peaks) * 0.8, max(b for b, _ in peaks) * 1.2)
        else:
            bt_topk = bt_wide
        # beat_this + oracle
        bt_oracle = dbn_btthis(bl, dl, gbpm * 0.8, gbpm * 1.2) if gbpm > 0 else bt_dbn
    
        row = {"track_id": tid, "gt_bpm": gbpm, "bt_bpm": bt_bpm,
               "tempo_class": tid_to_class[tid]}
        for label, beats in [("native", bt_raw), ("dbn55", bt_dbn), ("dbn30", bt_wide),
                              ("top1", bt_top1), ("topk", bt_topk), ("oracle", bt_oracle)]:
            r = eval_mireval(ref, beats)
            row[f"{label}_F"] = r["F"]; row[f"{label}_CMLt"] = r["CMLt"]; row[f"{label}_AMLt"] = r["AMLt"]
        cross_rows.append(row)
    
    # Summary
    print(f"\n{'System':<35} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
    print("=" * 63)
    for label, name in [("native", "beat_this (no DBN)"),
                         ("dbn55", "beat_this + DBN [55,215] (default)"),
                         ("dbn30", "beat_this + DBN [30,215] (wide)"),
                         ("top1", "beat_this + Beat Transformer tempo top-1 ±20%"),
                         ("topk", "beat_this + Beat Transformer tempo top-6 union"),
                         ("oracle", "beat_this + oracle (GT tempo)")]:
        f = np.mean([r[f"{label}_F"] for r in cross_rows])
        c = np.mean([r[f"{label}_CMLt"] for r in cross_rows])
        a = np.mean([r[f"{label}_AMLt"] for r in cross_rows])
        print(f"  {name:<33} {f:>8.4f} {c:>8.4f} {a:>8.4f}")

Cross-model:   0%|          | 0/217 [00:00<?, ?it/s]

Cross-model:   0%|          | 1/217 [00:00<02:17,  1.57it/s]

Cross-model:   1%|          | 2/217 [00:00<01:40,  2.13it/s]

Cross-model:   1%|▏         | 3/217 [00:01<01:40,  2.13it/s]

Cross-model:   2%|▏         | 4/217 [00:01<01:43,  2.06it/s]

...

Cross-model:  99%|█████████▉| 215/217 [01:40<00:00,  3.06it/s]

Cross-model: 100%|█████████▉| 216/217 [01:40<00:00,  3.12it/s]

Cross-model: 100%|██████████| 217/217 [01:40<00:00,  2.86it/s]

Cross-model: 100%|██████████| 217/217 [01:40<00:00,  2.15it/s]


System                                     F     CMLt     AMLt
  beat_this (no DBN)                  0.6270   0.5138   0.6100
  beat_this + DBN [55,215] (default)   0.5764   0.4738   0.6556
  beat_this + DBN [30,215] (wide)     0.5851   0.5102   0.6735
  beat_this + Beat Transformer tempo top-1 ±20%   0.5564   0.4762   0.6401
  beat_this + Beat Transformer tempo top-6 union   0.5871   0.5173   0.6722
  beat_this + oracle (GT tempo)       0.6179   0.7000   0.7290


### 7.7 Tempo Estimator Accuracy vs. Performance

How does tempo estimator accuracy (TCN 22% vs Beat Transformer 59% vs oracle 100%) translate into constraint effectiveness?


In [32]:
if not HAVE_BEAT_THIS:
    print("Skipping (beat_this data not available)")
else:
    # Load TCN results from prior experiment for comparison
    import csv
    tcn_csv = ROOT / "tcn_tempo_constrained_results.csv"
    tcn_rows = list(csv.DictReader(open(tcn_csv))) if tcn_csv.exists() else []
    
    if tcn_rows:
        tcn_correct = sum(1 for r in tcn_rows if r["tempo_class"] == "correct")
        tcn_constr_f = np.mean([float(r["bt_tcnconstr_F"]) for r in tcn_rows])
        tcn_dbn_f = np.mean([float(r["bt_dbn_F"]) for r in tcn_rows])
    else:
        tcn_correct = 48; tcn_constr_f = 0.434; tcn_dbn_f = 0.576
    
    bt_correct = sum(1 for r in tempo_rows if r["class"] == "correct")
    bt_constr_f = np.mean([r["top1_F"] for r in cross_rows])
    bt_dbn_f = np.mean([r["dbn55_F"] for r in cross_rows])
    oracle_f = np.mean([r["oracle_F"] for r in cross_rows])
    
    print("Three-Point Curve: Tempo Accuracy -> Constraint Effectiveness")
    print("(beat_this activations + DBNDownBeatTrackingProcessor)")
    print("=" * 75)
    print(f"  {'Estimator':<24} {'Correct':>14} {'bt+constr F':>14} {'bt+DBN F':>12}")
    print("-" * 75)
    print(f"  {'madmom TCN':<24} {tcn_correct}/217 ({100*tcn_correct/217:.0f}%)    {tcn_constr_f:>14.4f} {tcn_dbn_f:>12.4f}")
    print(f"  {'Beat Transformer':<24} {bt_correct}/217 ({100*bt_correct/217:.0f}%)   {bt_constr_f:>14.4f} {bt_dbn_f:>12.4f}")
    print(f"  {'oracle (GT tempo)':<24} {'217/217 (100%)':>14}    {oracle_f:>14.4f} {bt_dbn_f:>12.4f}")

Three-Point Curve: Tempo Accuracy -> Constraint Effectiveness
(beat_this activations + DBNDownBeatTrackingProcessor)
  Estimator                       Correct    bt+constr F     bt+DBN F
---------------------------------------------------------------------------
  madmom TCN               48/217 (22%)            0.4340       0.5764
  Beat Transformer         127/217 (59%)           0.5564       0.5764
  oracle (GT tempo)        217/217 (100%)            0.6179       0.5764


In [ ]:
# -- Figure 5: Tempo estimator accuracy vs. beat-tracking performance --
import matplotlib.pyplot as plt

if not HAVE_BEAT_THIS:
    print("Skipping Figure 5 (beat_this data not available)")
else:
    # Three-point curve uses the values computed in the cell above
    # Reconstruct the curve numerically from cross_rows / tcn_rows / tempo_rows
    bt_correct = sum(1 for r in tempo_rows if r["class"] == "correct")
    bt_constr_f    = np.mean([r["top1_F"]  for r in cross_rows])
    bt_constr_cmlt = np.mean([r["top1_CMLt"] for r in cross_rows])
    bt_constr_amlt = np.mean([r["top1_AMLt"] for r in cross_rows])
    oracle_f    = np.mean([r["oracle_F"]    for r in cross_rows])
    oracle_cmlt = np.mean([r["oracle_CMLt"] for r in cross_rows])
    oracle_amlt = np.mean([r["oracle_AMLt"] for r in cross_rows])

    # TCN point: load from tcn_tempo_constrained_results.csv if present
    tcn_csv = ROOT / "tcn_tempo_constrained_results.csv"
    if tcn_csv.exists():
        import csv as _csv
        _tcn_rows = list(_csv.DictReader(open(tcn_csv)))
        tcn_correct  = sum(1 for r in _tcn_rows if r["tempo_class"] == "correct")
        tcn_f    = np.mean([float(r["bt_tcnconstr_F"])    for r in _tcn_rows])
        tcn_cmlt = np.mean([float(r["bt_tcnconstr_CMLt"]) for r in _tcn_rows])
        tcn_amlt = np.mean([float(r["bt_tcnconstr_AMLt"]) for r in _tcn_rows])
    else:
        tcn_correct, tcn_f, tcn_cmlt, tcn_amlt = 48, 0.434, 0.201, 0.394

    xs = [100*tcn_correct/217, 100*bt_correct/217, 100.0]
    labels = [f"TCN\n({xs[0]:.0f}%)", f"Beat Trans.\n({xs[1]:.0f}%)", "GT tempo\n(100%)"]
    f_curve    = [tcn_f,    bt_constr_f,    oracle_f]
    cmlt_curve = [tcn_cmlt, bt_constr_cmlt, oracle_cmlt]
    amlt_curve = [tcn_amlt, bt_constr_amlt, oracle_amlt]

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(xs, f_curve,    'o-', color="#1f77b4", label="F-measure", linewidth=2, markersize=8)
    ax.plot(xs, cmlt_curve, 's-', color="#2ca02c", label="CMLt",     linewidth=2, markersize=8)
    ax.plot(xs, amlt_curve, '^-', color="#ff7f0e", label="AMLt",     linewidth=2, markersize=8)
    ax.axhline(y=0.627, color="#1f77b4", linestyle='--', alpha=0.5, linewidth=1)
    ax.text(xs[-1], 0.635, "Beat This raw F=0.627", fontsize=8, ha='right', color="#1f77b4")
    ax.axhline(y=0.514, color="#2ca02c", linestyle='--', alpha=0.5, linewidth=1)
    ax.text(xs[-1], 0.522, "Beat This raw CMLt=0.514", fontsize=8, ha='right', color="#2ca02c")

    ax.set_xticks(xs)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Tempo estimator accuracy on SMC (%)")
    ax.set_ylabel("Score")
    ax.set_title("Beat This activations + tempo-constrained DBN")
    ax.set_ylim(0.1, 0.8)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


### 7.8 GT-Tempo Effect Stratified by Error Category

Section 7.6 shows that GT-tempo lifts CMLt by +0.186 in aggregate but *reduces* F-measure slightly (-0.009). Does this pattern hold across all error categories, or does GT-tempo help some categories and hurt others? Figure 6 decomposes the effect by the error categories defined in §5.


In [ ]:
# -- Figure 6: GT-tempo DBN effect stratified by error category --
import matplotlib.pyplot as plt
import numpy as np

if not HAVE_BEAT_THIS:
    print("Skipping Figure 6 (beat_this data not available)")
else:
    # Attach oracle results (from cross_rows) to res_df for stratification
    _oracle_map = {r["track_id"]: r for r in cross_rows}
    res_df["_oracle_F"]    = [_oracle_map.get(tid, {}).get("oracle_F", np.nan)    for tid in res_df.index]
    res_df["_oracle_CMLt"] = [_oracle_map.get(tid, {}).get("oracle_CMLt", np.nan) for tid in res_df.index]

    cats = ["good", "octave_error", "continuity_drift", "total_failure", "other"]
    cat_labels = ["Good\n(F>=0.8)", "Octave\nerror", "Continuity\ndrift", "Total\nfailure", "Other"]

    raw_f, raw_cmlt, gt_f, gt_cmlt, ns = [], [], [], [], []
    for cat in cats:
        sub = res_df[res_df["error_cat"] == cat].dropna(subset=["_oracle_F"])
        ns.append(len(sub))
        raw_f.append(sub["F-measure"].mean())
        raw_cmlt.append(sub["CMLt"].mean())
        gt_f.append(sub["_oracle_F"].mean())
        gt_cmlt.append(sub["_oracle_CMLt"].mean())

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(cats))
    w = 0.2
    ax.bar(x - 1.5*w, raw_f,    w, label="F (raw)",        color="#1f77b4", edgecolor='black', linewidth=0.3)
    ax.bar(x - 0.5*w, gt_f,     w, label="F (GT-tempo)",   color="#7ec7ff", edgecolor='black', linewidth=0.3)
    ax.bar(x + 0.5*w, raw_cmlt, w, label="CMLt (raw)",     color="#2ca02c", edgecolor='black', linewidth=0.3)
    ax.bar(x + 1.5*w, gt_cmlt,  w, label="CMLt (GT-tempo)",color="#8fe38f", edgecolor='black', linewidth=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(cat_labels, ns)])
    ax.set_ylabel("Score")
    ax.set_title("Beat This activations + GT-tempo-constrained DBN")
    ax.set_ylim(0, 1.05)
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # Print the extreme case from the paper
    if "smc_064" in res_df.index:
        row = res_df.loc["smc_064"]
        print(f"smc_064 (extreme case): F raw={row['F-measure']:.3f}, "
              f"F GT-tempo={row['_oracle_F']:.3f}, CMLt GT-tempo={row['_oracle_CMLt']:.3f}")


### 7.9 DBN Parameter Sensitivity: Per-Track λ Sweep

All preceding DBN experiments kept `transition_lambda=100` (the madmom default). This parameter controls tempo smoothness: high values enforce rigid tempo continuity, low values let the DBN follow the activation more freely. If SMC's degraded activations demand a different tradeoff, the default λ may be systematically wrong.

We load results from a pre-computed sweep of `transition_lambda ∈ {1, 2, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500}` applied per-track to Beat This's cached activations with the wide DBN `[30, 215]`. For each track we retain the λ that maximizes F-measure. See `transition_lambda_sweep_report.md` for the full report.


In [ ]:
# -- DBN transition_lambda sweep: per-track optimal vs default --
# Data: transition_lambda_sweep_results.csv (pre-computed by scripts/transition_lambda_sweep.py)
import pandas as pd
import numpy as np
from collections import Counter

lam_df = pd.read_csv(ROOT / "transition_lambda_sweep_results.csv")
print(f"Loaded lambda sweep for {len(lam_df)} tracks")
print()

# Aggregate: default (lambda=100) vs per-track optimal
print("A. Aggregate Comparison (beat_this + wide DBN [30,215])")
print("=" * 60)
print(f"  {'Metric':<8} {'Default lam=100':>18} {'Per-track optimal':>20} {'Gap':>8}")
print("-" * 60)
print(f"  {'F':<8} {lam_df['default_F'].mean():>18.4f} {lam_df['optimal_F'].mean():>20.4f} "
      f"{lam_df['optimal_F'].mean() - lam_df['default_F'].mean():>+8.4f}")
print(f"  {'CMLt':<8} {lam_df['default_CMLt'].mean():>18.4f} {lam_df['optimal_CMLt'].mean():>20.4f} "
      f"{lam_df['optimal_CMLt'].mean() - lam_df['default_CMLt'].mean():>+8.4f}")
print(f"  {'AMLt':<8} {lam_df['default_AMLt'].mean():>18.4f} {lam_df['optimal_AMLt'].mean():>20.4f} "
      f"{lam_df['optimal_AMLt'].mean() - lam_df['default_AMLt'].mean():>+8.4f}")

# Globally optimal fixed lambda
lam_cols = [c for c in lam_df.columns if c.startswith("F_lam")]
lam_vals = [int(c.replace("F_lam", "")) for c in lam_cols]
mean_f_per_lam = [(lam, lam_df[c].mean()) for lam, c in zip(lam_vals, lam_cols)]
mean_f_per_lam.sort(key=lambda x: -x[1])
print()
print("B. Globally Optimal Fixed Lambda (top 5)")
print("=" * 60)
print(f"  {'lambda':>8} {'Mean F':>10}")
print("-" * 25)
for lam, f in mean_f_per_lam[:5]:
    marker = "  <-- best fixed lambda" if lam == mean_f_per_lam[0][0] else ""
    print(f"  {lam:>8} {f:>10.4f}{marker}")
print(f"\n  Default lambda=100 rank: #{1 + [lam for lam, _ in mean_f_per_lam].index(100)}")
print(f"  Default lambda=100 F:   {lam_df['F_lam100'].mean():.4f}")
print(f"  Best fixed lambda={mean_f_per_lam[0][0]}:  {mean_f_per_lam[0][1]:.4f}")
print(f"  Best fixed gain:        {mean_f_per_lam[0][1] - lam_df['F_lam100'].mean():+.4f}")

# Distribution of per-track optimal lambdas (bimodality check)
print()
print("C. Distribution of Per-Track Optimal Lambdas")
print("=" * 60)
lam_counts = Counter(lam_df["optimal_lambda"])
for lam in sorted(lam_counts):
    n = lam_counts[lam]
    bar = "#" * int(50 * n / len(lam_df))
    marker = "  <-- mode" if n == max(lam_counts.values()) else ""
    print(f"  lam={lam:>4}  n={n:>3} ({100*n/len(lam_df):>4.1f}%)  {bar}{marker}")

# By error category
print()
print("D. Optimal Lambda by Error Category")
print("=" * 60)
print(f"  {'Category':<18} {'N':>4} {'Median lam*':>12} {'Mean F (default)':>18} {'Mean F (opt)':>14}")
print("-" * 70)
for cat in ["good", "octave_error", "continuity_drift", "total_failure", "other"]:
    sub = lam_df[lam_df["error_category"] == cat]
    if len(sub) == 0:
        continue
    print(f"  {cat:<18} {len(sub):>4} {sub['optimal_lambda'].median():>12.0f} "
          f"{sub['default_F'].mean():>18.4f} {sub['optimal_F'].mean():>14.4f}")

print()
print("Key finding: one-third of tracks (mostly 'good' category) prefer lam=1,")
print("meaning the DBN's tempo prior actively hurts them. The fixed-parameter DBN")
print("cannot simultaneously serve tracks with clean activations and tracks with noisy ones.")


### 7.10 Compounding Optimal λ with GT-Tempo (Table 3)

The per-track λ sweep recovers +0.050 F from the decoder. The GT-tempo constraint recovers +0.186 CMLt from metrical coherence. A natural question is whether these gains are independent or interacting: if we combine both — the correct tempo *and* a per-track optimal λ — do we get the sum, or is there destructive interference? We load the combined sweep results (`gt_tempo_lambda_sweep_results.csv`) to answer this.


In [ ]:
# -- Table 3: Beat This configurations on SMC (compounding optimal lambda + GT-tempo) --
import pandas as pd
import numpy as np

gt_lam_df = pd.read_csv(ROOT / "gt_tempo_lambda_sweep_results.csv")
lam_df    = pd.read_csv(ROOT / "transition_lambda_sweep_results.csv")

# Row 1: Peak-picking (no DBN) — from res_df
peak_f    = res_df["F-measure"].mean()
peak_cmlt = res_df["CMLt"].mean()
peak_amlt = res_df["AMLt"].mean()

# Row 2a: Narrow DBN [55,215] lam=100 — the library default, from res_df dbn_* columns
narrow_f    = res_df["dbn_F-measure"].mean()
narrow_cmlt = res_df["dbn_CMLt"].mean()
narrow_amlt = res_df["dbn_AMLt"].mean()

# Row 2b: Wide DBN [30,215] lam=100 — the baseline for the per-track lambda sweep
wide_f    = lam_df["default_F"].mean()
wide_cmlt = lam_df["default_CMLt"].mean()
wide_amlt = lam_df["default_AMLt"].mean()

# Row 3: Wide DBN per-track optimal lam — from lam_df
dbn_opt_f    = lam_df["optimal_F"].mean()
dbn_opt_cmlt = lam_df["optimal_CMLt"].mean()
dbn_opt_amlt = lam_df["optimal_AMLt"].mean()

# Row 4: GT-tempo fixed lam=100 — from gt_lam_df
gt100_f    = gt_lam_df["default_lambda_F"].mean()
gt100_cmlt = gt_lam_df["default_lambda_CMLt"].mean()
gt100_amlt = gt_lam_df["default_lambda_AMLt"].mean()

# Row 5: GT-tempo per-track optimal lam — from gt_lam_df
gt_opt_f    = gt_lam_df["optimal_F"].mean()
gt_opt_cmlt = gt_lam_df["optimal_CMLt"].mean()
gt_opt_amlt = gt_lam_df["optimal_AMLt"].mean()

# Row 6: GT activations + DBN (wide, lam=100) — from the Experiment 2 per-track list
# (gt_dbn30_rows is populated in the Experiment 2 cell in §7.5 and cached on disk)
gt_act_f = gt_act_cmlt = gt_act_amlt = float("nan")
try:
    gt_act_csv = ROOT / "gt_activation_dbn_per_track.csv"
    if gt_act_csv.exists():
        _gt_act = pd.read_csv(gt_act_csv)
        gt_act_f    = _gt_act["F"].mean()
        gt_act_cmlt = _gt_act["CMLt"].mean()
        gt_act_amlt = _gt_act["AMLt"].mean()
except Exception as _e:
    pass

print("Table 3: Beat This configurations on SMC")
print("=" * 72)
print(f"  {'Configuration':<38} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
print("-" * 72)
print(f"  {'Peak-picking (no DBN)':<38} {peak_f:>8.3f} {peak_cmlt:>8.3f} {peak_amlt:>8.3f}")
print(f"  {'DBN lam=100 (narrow [55,215])':<38} {narrow_f:>8.3f} {narrow_cmlt:>8.3f} {narrow_amlt:>8.3f}")
print(f"  {'DBN lam=100 (wide [30,215])':<38} {wide_f:>8.3f} {wide_cmlt:>8.3f} {wide_amlt:>8.3f}")
print(f"  {'DBN optimal lam (wide)':<38} {dbn_opt_f:>8.3f} {dbn_opt_cmlt:>8.3f} {dbn_opt_amlt:>8.3f}")
print(f"  {'GT-tempo DBN lam=100':<38} {gt100_f:>8.3f} {gt100_cmlt:>8.3f} {gt100_amlt:>8.3f}")
print(f"  {'GT-tempo + optimal lam':<38} {gt_opt_f:>8.3f} {gt_opt_cmlt:>8.3f} {gt_opt_amlt:>8.3f}")
if not np.isnan(gt_act_f):
    print(f"  {'GT activations + DBN (wide)':<38} {gt_act_f:>8.3f} {gt_act_cmlt:>8.3f} {gt_act_amlt:>8.3f}")
print()

# Compounding analysis (relative to wide DBN baseline, matching §5.4.6 narrative)
gt_gain_alone  = gt100_f  - wide_f
lam_gain_alone = dbn_opt_f - wide_f
combined_gain  = gt_opt_f  - wide_f
interaction    = combined_gain - (gt_gain_alone + lam_gain_alone)

print("Compounding Analysis (relative to wide DBN lam=100)")
print("=" * 72)
print(f"  GT-tempo gain alone (fixed lam=100): {gt_gain_alone:>+.4f}")
print(f"  Optimal-lam gain alone (wide DBN):   {lam_gain_alone:>+.4f}")
print(f"  Sum if independent:                  {gt_gain_alone + lam_gain_alone:>+.4f}")
print(f"  Actual combined gain:                {combined_gain:>+.4f}")
print(f"  Interaction term:                    {interaction:>+.4f}")
print()
print(f"  GT-tempo + optimal lam reaches F={gt_opt_f:.3f} — the first DBN configuration")
print(f"  that surpasses Beat This raw peak-picking (F={peak_f:.3f}) AND lifts CMLt to {gt_opt_cmlt:.3f}.")


## 8. Cross-System Comparison (madmom TCN)

We compare **madmom's TCN+DBN beat tracker** (Böck & Davies, 2019/2020) against beat_this as an independent baseline. This is the stronger of madmom's two beat trackers and is widely used as a reference.


In [33]:
# ── Load madmom TCN results ──
import csv

TCN_OUT_DIR = ROOT / "madmom_tcn_output"
TCN_CSV = ROOT / "tcn_tempo_constrained_results.csv"

tcn_rows = list(csv.DictReader(open(TCN_CSV)))
tcn_by_tid = {r["track_id"]: r for r in tcn_rows}
print(f"Loaded {len(tcn_rows)} tracks from TCN results")

# ── Aggregate comparison ──
mm_f = np.mean([float(r["mm_tcn_F"]) for r in tcn_rows])
mm_c = np.mean([float(r["mm_tcn_CMLt"]) for r in tcn_rows])
mm_a = np.mean([float(r["mm_tcn_AMLt"]) for r in tcn_rows])
bt_f = np.mean([float(r["bt_F"]) for r in tcn_rows])
bt_c = np.mean([float(r["bt_CMLt"]) for r in tcn_rows])
bt_a = np.mean([float(r["bt_AMLt"]) for r in tcn_rows])
bt_dbn_f = np.mean([float(r["bt_dbn_F"]) for r in tcn_rows])
bt_dbn_c = np.mean([float(r["bt_dbn_CMLt"]) for r in tcn_rows])
bt_dbn_a = np.mean([float(r["bt_dbn_AMLt"]) for r in tcn_rows])

print(f"\n{'System':<30} {'Mean F':>8} {'CMLt':>8} {'AMLt':>8}")
print("=" * 58)
print(f"  {'madmom TCN+DBN':<28} {mm_f:>8.4f} {mm_c:>8.4f} {mm_a:>8.4f}")
print(f"  {'beat_this (no DBN)':<28} {bt_f:>8.4f} {bt_c:>8.4f} {bt_a:>8.4f}")
print(f"  {'beat_this + DBN [55,215]':<28} {bt_dbn_f:>8.4f} {bt_dbn_c:>8.4f} {bt_dbn_a:>8.4f}")
print(f"\n  beat_this wins overall on F-measure even though madmom likely saw SMC during training.")

Loaded 217 tracks from TCN results

System                           Mean F     CMLt     AMLt
  madmom TCN+DBN                 0.5895   0.4762   0.6632
  beat_this (no DBN)             0.6270   0.5138   0.6100
  beat_this + DBN [55,215]       0.5764   0.4738   0.6556

  beat_this wins overall on F-measure even though madmom likely saw SMC during training.


In [34]:
# ── Per-track win analysis ──
mm_wins = sum(1 for r in tcn_rows if float(r["mm_tcn_F"]) > float(r["bt_F"]))
bt_wins = sum(1 for r in tcn_rows if float(r["bt_F"]) > float(r["mm_tcn_F"]))
ties = len(tcn_rows) - mm_wins - bt_wins
print(f"Per-track wins (by F-measure): madmom TCN={mm_wins}, beat_this={bt_wins}, ties={ties}")

# Tracks where madmom TCN wins big (delta > 0.1)
mm_wins_big = [(r, float(r["mm_tcn_F"]) - float(r["bt_F"]))
               for r in tcn_rows if float(r["mm_tcn_F"]) - float(r["bt_F"]) > 0.1]
bt_wins_big = [(r, float(r["bt_F"]) - float(r["mm_tcn_F"]))
               for r in tcn_rows if float(r["bt_F"]) - float(r["mm_tcn_F"]) > 0.1]

print(f"\nTracks where madmom TCN >> beat_this (delta F > 0.1): {len(mm_wins_big)}")
for r, d in sorted(mm_wins_big, key=lambda x: -x[1])[:10]:
    print(f"  {r['track_id']}: mm={float(r['mm_tcn_F']):.3f}  bt={float(r['bt_F']):.3f}  "
          f"delta={d:+.3f}  (gt={float(r['gt_bpm']):.1f} BPM)")

print(f"\nTracks where beat_this >> madmom TCN (delta F > 0.1): {len(bt_wins_big)}")
for r, d in sorted(bt_wins_big, key=lambda x: -x[1])[:10]:
    print(f"  {r['track_id']}: mm={float(r['mm_tcn_F']):.3f}  bt={float(r['bt_F']):.3f}  "
          f"delta={d:+.3f}  (gt={float(r['gt_bpm']):.1f} BPM)")

Per-track wins (by F-measure): madmom TCN=78, beat_this=122, ties=17

Tracks where madmom TCN >> beat_this (delta F > 0.1): 38
  smc_285: mm=1.000  bt=0.354  delta=+0.646  (gt=120.0 BPM)
  smc_153: mm=0.893  bt=0.313  delta=+0.580  (gt=96.2 BPM)
  smc_059: mm=0.916  bt=0.375  delta=+0.541  (gt=71.4 BPM)
  smc_109: mm=0.842  bt=0.407  delta=+0.435  (gt=65.5 BPM)
  smc_064: mm=0.342  bt=0.000  delta=+0.342  (gt=44.2 BPM)
  smc_028: mm=0.763  bt=0.425  delta=+0.338  (gt=117.2 BPM)
  smc_076: mm=0.787  bt=0.465  delta=+0.323  (gt=86.9 BPM)
  smc_181: mm=0.657  bt=0.336  delta=+0.321  (gt=61.7 BPM)
  smc_027: mm=0.838  bt=0.568  delta=+0.270  (gt=63.8 BPM)
  smc_176: mm=0.609  bt=0.343  delta=+0.266  (gt=80.7 BPM)

Tracks where beat_this >> madmom TCN (delta F > 0.1): 71
  smc_215: mm=0.371  bt=0.879  delta=+0.507  (gt=57.3 BPM)
  smc_030: mm=0.465  bt=0.941  delta=+0.476  (gt=200.0 BPM)
  smc_207: mm=0.538  bt=0.955  delta=+0.417  (gt=113.7 BPM)
  smc_022: mm=0.543  bt=0.949  delta=+0.405 

In [35]:
# ── IBI / Metrical level analysis ──
# Load madmom TCN beat predictions and compute IBI ratios
mm_correct_ibi = 0
bt_correct_ibi = 0
both_correct = 0
both_wrong = 0
mm_correct_bt_wrong = 0
mm_wrong_bt_correct = 0

mm_wins_ibi_details = []  # for the tracks where madmom wins big
mm_wins_big_tids = {r['track_id'] for r, _ in mm_wins_big}

for r in tcn_rows:
    tid = r['track_id']
    gt_bpm = float(r['gt_bpm'])
    if gt_bpm <= 0:
        continue
    gt_ibi = 60.0 / gt_bpm

    # madmom TCN IBI
    mm_path = TCN_OUT_DIR / f"{tid}.beats"
    mm_beats = np.loadtxt(str(mm_path))
    if mm_beats.ndim == 0:
        mm_beats = np.array([float(mm_beats)])
    mm_ibi = float(np.median(np.diff(mm_beats))) if len(mm_beats) > 1 else 0

    # beat_this IBI
    bt_path = ROOT / "beat_this_output" / f"{tid.upper()}.beats"
    if bt_path.exists():
        bt_beats = np.array([float(l.split('\t')[0]) for l in open(bt_path) if l.strip()])
    else:
        bt_beats = np.array([])
    bt_ibi = float(np.median(np.diff(bt_beats))) if len(bt_beats) > 1 else 0

    mm_ratio = mm_ibi / gt_ibi if gt_ibi > 0 and mm_ibi > 0 else 0
    bt_ratio = bt_ibi / gt_ibi if gt_ibi > 0 and bt_ibi > 0 else 0
    mm_ok = 0.7 <= mm_ratio <= 1.4
    bt_ok = 0.7 <= bt_ratio <= 1.4

    if mm_ok: mm_correct_ibi += 1
    if bt_ok: bt_correct_ibi += 1
    if mm_ok and bt_ok: both_correct += 1
    elif not mm_ok and not bt_ok: both_wrong += 1
    elif mm_ok and not bt_ok: mm_correct_bt_wrong += 1
    else: mm_wrong_bt_correct += 1

    if tid in mm_wins_big_tids:
        mm_wins_ibi_details.append((tid, mm_ratio, bt_ratio, mm_ok))

print("Metrical Level (IBI ratio) Analysis")
print("=" * 55)
print(f"  madmom TCN correct metrical level: {mm_correct_ibi}/217 ({100*mm_correct_ibi/217:.0f}%)")
print(f"  beat_this correct metrical level:  {bt_correct_ibi}/217 ({100*bt_correct_ibi/217:.0f}%)")

print(f"\nCross-system metrical level agreement:")
print(f"  Both correct:                     {both_correct}")
print(f"  Both wrong:                       {both_wrong}")
print(f"  madmom correct, beat_this wrong:  {mm_correct_bt_wrong}")
print(f"  beat_this correct, madmom wrong:  {mm_wrong_bt_correct}")

# When madmom wins big, how often does it have the right tempo?
mm_wins_correct_ibi = sum(1 for _, _, _, ok in mm_wins_ibi_details if ok)
print(f"\nAmong {len(mm_wins_big)} tracks where madmom TCN wins big:")
print(f"  Correct metrical level: {mm_wins_correct_ibi}/{len(mm_wins_big)} "
      f"({100*mm_wins_correct_ibi/len(mm_wins_big):.0f}%)")
print(f"  → madmom's DBN locks onto the right metrical level on most of these tracks.")

Metrical Level (IBI ratio) Analysis
  madmom TCN correct metrical level: 138/217 (64%)
  beat_this correct metrical level:  142/217 (65%)

Cross-system metrical level agreement:
  Both correct:                     106
  Both wrong:                       43
  madmom correct, beat_this wrong:  32
  beat_this correct, madmom wrong:  36

Among 38 tracks where madmom TCN wins big:
  Correct metrical level: 36/38 (95%)
  → madmom's DBN locks onto the right metrical level on most of these tracks.


In [36]:
# ── Complementarity: cross-system oracle ──
oracle_f = [max(float(r["mm_tcn_F"]), float(r["bt_F"])) for r in tcn_rows]
oracle_c = [max(float(r["mm_tcn_CMLt"]), float(r["bt_CMLt"])) for r in tcn_rows]
oracle_a = [max(float(r["mm_tcn_AMLt"]), float(r["bt_AMLt"])) for r in tcn_rows]

print("Cross-System Oracle (per-track best of beat_this vs madmom TCN)")
print("=" * 65)
print(f"  {'System':<35} {'F':>8} {'CMLt':>8} {'AMLt':>8}")
print("-" * 65)
print(f"  {'beat_this (no DBN)':<35} {np.mean([float(r['bt_F']) for r in tcn_rows]):>8.4f} "
      f"{np.mean([float(r['bt_CMLt']) for r in tcn_rows]):>8.4f} "
      f"{np.mean([float(r['bt_AMLt']) for r in tcn_rows]):>8.4f}")
print(f"  {'madmom TCN+DBN':<35} {np.mean([float(r['mm_tcn_F']) for r in tcn_rows]):>8.4f} "
      f"{np.mean([float(r['mm_tcn_CMLt']) for r in tcn_rows]):>8.4f} "
      f"{np.mean([float(r['mm_tcn_AMLt']) for r in tcn_rows]):>8.4f}")
print(f"  {'Oracle (per-track best)':<35} {np.mean(oracle_f):>8.4f} "
      f"{np.mean(oracle_c):>8.4f} {np.mean(oracle_a):>8.4f}")
print(f"\n  Oracle gain over beat_this: +{np.mean(oracle_f) - np.mean([float(r['bt_F']) for r in tcn_rows]):.3f} F")
print(f"  → The two systems are complementary: combining them per-track yields")
print(f"    a meaningful improvement, even though neither system dominates overall.")

Cross-System Oracle (per-track best of beat_this vs madmom TCN)
  System                                     F     CMLt     AMLt
-----------------------------------------------------------------
  beat_this (no DBN)                    0.6270   0.5138   0.6100
  madmom TCN+DBN                        0.5895   0.4762   0.6632
  Oracle (per-track best)               0.6738   0.6305   0.7269

  Oracle gain over beat_this: +0.047 F
  → The two systems are complementary: combining them per-track yields
    a meaningful improvement, even though neither system dominates overall.


In [37]:
# ── Complementarity: what explains madmom's wins? ──
# Among tracks where madmom has correct IBI, who wins?
mm_correct_tracks = []
for r in tcn_rows:
    gt_bpm = float(r['gt_bpm'])
    if gt_bpm <= 0:
        continue
    gt_ibi = 60.0 / gt_bpm
    mm_path = TCN_OUT_DIR / f"{r['track_id']}.beats"
    mm_beats = np.loadtxt(str(mm_path))
    if mm_beats.ndim == 0:
        mm_beats = np.array([float(mm_beats)])
    mm_ibi = float(np.median(np.diff(mm_beats))) if len(mm_beats) > 1 else 0
    if gt_ibi > 0 and mm_ibi > 0 and 0.7 <= mm_ibi / gt_ibi <= 1.4:
        mm_correct_tracks.append(r)

n = len(mm_correct_tracks)
mm_wins_correct = sum(1 for r in mm_correct_tracks if float(r['mm_tcn_F']) > float(r['bt_F']))
bt_wins_correct = sum(1 for r in mm_correct_tracks if float(r['bt_F']) > float(r['mm_tcn_F']))
ties_correct = n - mm_wins_correct - bt_wins_correct

print(f"Among {n} tracks where madmom TCN has correct metrical level:")
print(f"  madmom wins: {mm_wins_correct} ({100*mm_wins_correct/n:.0f}%)")
print(f"  beat_this wins: {bt_wins_correct} ({100*bt_wins_correct/n:.0f}%)")
print(f"  ties: {ties_correct} ({100*ties_correct/n:.0f}%)")
print(f"\n  → Correct tempo is necessary but not sufficient for madmom to win.")
print(f"    Even with the right metrical level, beat_this's activation quality")
print(f"    gives it the edge on many tracks. Beat tracking depends on two")
print(f"    independent factors: activation quality and metrical level selection.")

Among 138 tracks where madmom TCN has correct metrical level:
  madmom wins: 69 (50%)
  beat_this wins: 55 (40%)
  ties: 14 (10%)

  → Correct tempo is necessary but not sufficient for madmom to win.
    Even with the right metrical level, beat_this's activation quality
    gives it the edge on many tracks. Beat tracking depends on two
    independent factors: activation quality and metrical level selection.


## 9. Discussion & Summary

We close with the two-ceilings framing, a comprehensive summary table across all systems and configurations, and a list of key findings.


### 9.1 Two Performance Ceilings

Our experiments converge on two independent performance ceilings on SMC:

1. **Activation ceiling (~F = 0.67)** — The maximum F-measure achievable by any DBN decoder operating on the real activations. It exists because ~100 tracks produce confident activation peaks at wrong positions that no decoder can override. Evidence: per-track optimal λ + GT-tempo hits F=0.667; GT activations + DBN hits F=0.924.

2. **Tempo ceiling (~CMLt = 0.73, ~AMLt = 0.76)** — The maximum metrical coherence achievable with perfect tempo through the current DBN. Evidence: GT-tempo + optimal λ hits CMLt=0.735 / AMLt=0.755.

This cell prints the best configurations under each regime, which reproduces the numbers cited in the paper's Discussion §6.1.

In [ ]:
# -- Two performance ceilings on SMC --
#
# Activation ceiling: the maximum F achievable by any decoder on the real activations.
# Tempo ceiling: the maximum metrical coherence (CMLt/AMLt) achievable with perfect tempo.
import pandas as pd
import numpy as np

gt_lam_df = pd.read_csv(ROOT / "gt_tempo_lambda_sweep_results.csv")
lam_df    = pd.read_csv(ROOT / "transition_lambda_sweep_results.csv")

print("Two Performance Ceilings on SMC")
print("=" * 75)
print()
print("1. ACTIVATION CEILING — max F-measure any decoder can reach on real activations")
print("-" * 75)
print(f"  Beat This raw peak-picking:           F = {res_df['F-measure'].mean():.4f}")
print(f"  Wide DBN lam=100:                     F = {lam_df['default_F'].mean():.4f}")
print(f"  Wide DBN per-track optimal lam:       F = {lam_df['optimal_F'].mean():.4f}")
print(f"  GT-tempo DBN per-track optimal lam:   F = {gt_lam_df['optimal_F'].mean():.4f}  <-- best real-activation F")
print(f"  --> Activation ceiling: F ~ 0.67")
print()
print("  Reference: GT activations + DBN        F = 0.924  (SMC, from cross-dataset table)")
print("  The ~0.25 F gap between real and synthetic activations is the cost of")
print("  confident-but-wrong activation peaks that no decoder can override.")
print()

print("2. TEMPO CEILING — max metrical coherence with perfect tempo injected into the DBN")
print("-" * 75)
print(f"  Beat This raw peak-picking:           CMLt = {res_df['CMLt'].mean():.4f}   AMLt = {res_df['AMLt'].mean():.4f}")
print(f"  GT-tempo DBN per-track optimal lam:   CMLt = {gt_lam_df['optimal_CMLt'].mean():.4f}   AMLt = {gt_lam_df['optimal_AMLt'].mean():.4f}")
print(f"  --> Tempo ceiling: CMLt ~ 0.73, AMLt ~ 0.76")
print()

print("3. WHAT EACH DIFFICULTY AXIS HITS")
print("-" * 75)
print("  Metrical ambiguity tracks    --> tempo ceiling (largest Delta CMLt from GT tempo)")
print("  Tempo instability tracks     --> both ceilings (degraded activations AND mis-tempo)")
print("  Weak beat cues               --> neither ceiling dominates; rho(axis, act_q) ~ 0")
print("  Structural/context           --> moderate activation degradation only")


In [38]:
if not HAVE_BEAT_THIS:
    cross_rows = []

# Build comprehensive summary DataFrame
summary_data = []

# Beat-Transformer-on-itself systems
for label, name in [("paper", "Beat Transformer + DBN [55,215] (paper)"),
                     ("wide", "Beat Transformer + DBN [30,215] (wide)"),
                     ("top1", "Beat Transformer + tempo top-1"),
                     ("topk", "Beat Transformer + tempo top-6 union"),
                     ("viterbi", "Beat Transformer + Viterbi-select"),
                     ("oracle", "Beat Transformer + oracle (GT tempo)")]:
    summary_data.append({
        "System": name,
        "F": round(np.mean([r[f"{label}_F"] for r in bt_on_bt_rows]), 4),
        "CMLt": round(np.mean([r[f"{label}_CMLt"] for r in bt_on_bt_rows]), 4),
        "AMLt": round(np.mean([r[f"{label}_AMLt"] for r in bt_on_bt_rows]), 4),
    })

# madmom TCN+DBN baseline
summary_data.append({
    "System": "madmom TCN+DBN",
    "F": round(np.mean([float(r["mm_tcn_F"]) for r in tcn_rows]), 4),
    "CMLt": round(np.mean([float(r["mm_tcn_CMLt"]) for r in tcn_rows]), 4),
    "AMLt": round(np.mean([float(r["mm_tcn_AMLt"]) for r in tcn_rows]), 4),
})

# beat_this systems
for label, name in [("native", "beat_this (no DBN)"),
                     ("dbn55", "beat_this + DBN [55,215]"),
                     ("dbn30", "beat_this + DBN [30,215]"),
                     ("top1", "beat_this + Beat Transformer tempo top-1"),
                     ("topk", "beat_this + Beat Transformer tempo top-6 union"),
                     ("oracle", "beat_this + oracle (GT tempo)")]:
    summary_data.append({
        "System": name,
        "F": round(np.mean([r[f"{label}_F"] for r in cross_rows]), 4),
        "CMLt": round(np.mean([r[f"{label}_CMLt"] for r in cross_rows]), 4),
        "AMLt": round(np.mean([r[f"{label}_AMLt"] for r in cross_rows]), 4),
    })

# Oracle ceilings
summary_data.append({
    "System": "Oracle (bt / mm_tcn per-track best)",
    "F": round(np.mean([max(float(r["mm_tcn_F"]), float(r["bt_F"])) for r in tcn_rows]), 4),
    "CMLt": round(np.mean([max(float(r["mm_tcn_CMLt"]), float(r["bt_CMLt"])) for r in tcn_rows]), 4),
    "AMLt": round(np.mean([max(float(r["mm_tcn_AMLt"]), float(r["bt_AMLt"])) for r in tcn_rows]), 4),
})

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Export
summary_df.to_csv("bt_transformer_aggregate_summary.csv", index=False)
print(f"\nSaved to bt_transformer_aggregate_summary.csv")

                                        System      F   CMLt   AMLt
       Beat Transformer + DBN [55,215] (paper) 0.5656 0.4639 0.6316
        Beat Transformer + DBN [30,215] (wide) 0.5805 0.5232 0.6433
                Beat Transformer + tempo top-1 0.5670 0.4927 0.6348
          Beat Transformer + tempo top-6 union 0.5802 0.5224 0.6433
             Beat Transformer + Viterbi-select 0.5818 0.5277 0.6458
          Beat Transformer + oracle (GT tempo) 0.6267 0.7123 0.7263
                                madmom TCN+DBN 0.5895 0.4762 0.6632
                            beat_this (no DBN) 0.6270 0.5138 0.6100
                      beat_this + DBN [55,215] 0.5764 0.4738 0.6556
                      beat_this + DBN [30,215] 0.5851 0.5102 0.6735
      beat_this + Beat Transformer tempo top-1 0.5564 0.4762 0.6401
beat_this + Beat Transformer tempo top-6 union 0.5871 0.5173 0.6722
                 beat_this + oracle (GT tempo) 0.6179 0.7000 0.7290
           Oracle (bt / mm_tcn per-track best) 0

In [39]:
if not HAVE_BEAT_THIS:
    print("Skipping per-track export (no beat_this data)")
else:
    # Export per-track results to CSV
    all_rows = []
    for i, tid in enumerate(track_ids):
        row = {
            "track_id": tid,
            "gt_bpm": round(tempo_rows[i]["gt_bpm"], 2),
            "bt_transformer_bpm": round(tempo_rows[i]["bt_bpm"], 2),
            "tempo_ratio": round(tempo_rows[i]["ratio"], 4),
            "tempo_class": tempo_rows[i]["class"],
        }
        # Beat-Transformer-on-itself columns
        btr = bt_on_bt_rows[i]
        for label in ["paper", "wide", "top1", "topk", "viterbi", "oracle"]:
            for m in ["F", "CMLt", "AMLt"]:
                row[f"bt_{label}_{m}"] = round(btr[f"{label}_{m}"], 4)
        # Cross-model columns
        cr = cross_rows[i]
        for label in ["native", "dbn55", "dbn30", "top1", "topk", "oracle"]:
            for m in ["F", "CMLt", "AMLt"]:
                row[f"btthis_{label}_{m}"] = round(cr[f"{label}_{m}"], 4)
        # madmom TCN columns
        tcn_r = tcn_by_tid.get(tid, {})
        if tcn_r:
            row["mm_tcn_F"] = round(float(tcn_r["mm_tcn_F"]), 4)
            row["mm_tcn_CMLt"] = round(float(tcn_r["mm_tcn_CMLt"]), 4)
            row["mm_tcn_AMLt"] = round(float(tcn_r["mm_tcn_AMLt"]), 4)
        all_rows.append(row)
    
    results_df = pd.DataFrame(all_rows)
    results_df.to_csv("bt_transformer_tempo_constrained_results.csv", index=False)
    print(f"Per-track CSV: {len(results_df)} rows x {len(results_df.columns)} columns")
    print(f"Columns: {list(results_df.columns)}")

Per-track CSV: 217 rows x 44 columns
Columns: ['track_id', 'gt_bpm', 'bt_transformer_bpm', 'tempo_ratio', 'tempo_class', 'bt_paper_F', 'bt_paper_CMLt', 'bt_paper_AMLt', 'bt_wide_F', 'bt_wide_CMLt', 'bt_wide_AMLt', 'bt_top1_F', 'bt_top1_CMLt', 'bt_top1_AMLt', 'bt_topk_F', 'bt_topk_CMLt', 'bt_topk_AMLt', 'bt_viterbi_F', 'bt_viterbi_CMLt', 'bt_viterbi_AMLt', 'bt_oracle_F', 'bt_oracle_CMLt', 'bt_oracle_AMLt', 'btthis_native_F', 'btthis_native_CMLt', 'btthis_native_AMLt', 'btthis_dbn55_F', 'btthis_dbn55_CMLt', 'btthis_dbn55_AMLt', 'btthis_dbn30_F', 'btthis_dbn30_CMLt', 'btthis_dbn30_AMLt', 'btthis_top1_F', 'btthis_top1_CMLt', 'btthis_top1_AMLt', 'btthis_topk_F', 'btthis_topk_CMLt', 'btthis_topk_AMLt', 'btthis_oracle_F', 'btthis_oracle_CMLt', 'btthis_oracle_AMLt', 'mm_tcn_F', 'mm_tcn_CMLt', 'mm_tcn_AMLt']


### 9.2 Key Findings

1. **Paper replication**: Beat Transformer F=0.596 on SMC reproduced exactly (madmom + quantized GT protocol). The 0.03 gap vs mir_eval is fully explained by `trim_beats` (removes beats < 5s) and GT quantization (rounds to 43 fps grid).

2. **SMC difficulty is combinatorial**: Hard tracks average 3.85 tags and 2.33 active difficulty axes (weak beat cues, tempo instability, metrical ambiguity, structural/context). 84% of hard tracks activate 2+ axes simultaneously.

3. **beat_this baseline**: F=0.627, with three distinct failure modes -- octave errors (8%), continuity drift (18%), and total failure (5%). Performance degrades monotonically with annotator confidence (1: F=0.722, 4: F=0.421) and number of descriptors.

4. **DBN hurts F-measure**: Adding DBN postprocessing drops F from 0.627 to 0.576 (-0.051), worsening more tracks than it helps. DBN confidence (Viterbi log-prob) fails as a per-track selector -- helped and hurt distributions overlap.

5. **Confident-but-wrong activations**: The dominant failure mode is strong activation peaks at wrong positions, not absent activations. Max activation barely drops (0.998 to 0.931) but activation at GT positions drops 3.2x (0.915 to 0.284). `bt_act_gt_peak_mean` is the strongest predictor of F-measure (rho=+0.784).

6. **All post-processing interventions fail**: ACF octave correction, IBI outlier cleanup, threshold sweep. The activation quality ceiling cannot be broken at inference time.

7. **Wide-DBN improvement**: Changing `min_bpm` from 55 to 30 gives free gains: Beat Transformer +0.015 F/+0.059 CMLt, beat_this +0.009 F/+0.036 CMLt. One-line fix, zero retraining.

8. **Tempo head accuracy**: Beat Transformer's tempo head is 59% correct on SMC (vs madmom TCN's 22%). Accuracy varies by BPM range: 85% at 70-90 BPM but only 31% below 55 BPM.

9. **Tempo constraint asymmetry**: Correct constraints help but wrong constraints destroy. The 41% error rate makes aggregate-positive hard constraint impossible. Multi-hypothesis (top-6 union) outperforms single-point.

10. **Per-track optimal lambda**: Sweeping `transition_lambda` per track recovers F=0.642 (+0.050 over default), the first DBN configuration that surpasses Beat This raw peak-picking. The distribution is bimodal: 33% of tracks prefer lambda=1 (minimal smoothing), while failing tracks prefer higher values. No single lambda can serve both populations.

11. **Two ceilings**: The activation ceiling (~F=0.67) comes from confident-but-wrong peaks; the tempo ceiling (~CMLt=0.73) comes from the DBN's rigid tempo prior. GT-tempo + optimal lambda reaches F=0.667 / CMLt=0.735, approaching both ceilings simultaneously but requiring oracle knowledge per track.

12. **beat_this vs madmom TCN**: beat_this (F=0.627) outperforms madmom TCN+DBN (F=0.589) despite madmom likely training on SMC. Per-track oracle: F=0.674. Correct tempo is necessary but not sufficient for madmom to win.

13. **Training data mismatch**: Training sets are overwhelmingly percussion-heavy, steady-tempo (80-140 BPM). SMC median is 71 BPM with 28% below 60 BPM. Zero acapella training data. This drives both the activation quality gap and tempo estimation failures.


## 10. Paper-numbers audit

Every numeric claim in the paper that isn't already printed by an earlier cell is reproduced here from the cached CSVs, so `nbconvert --execute` verifies the whole paper in one pass.


In [ ]:
# -- Paper-numbers audit --
# Reproduces every paper claim not already printed by earlier cells.
import pandas as pd
import numpy as np

btc      = pd.read_csv(ROOT/"bt_transformer_tempo_constrained_results.csv").set_index("track_id")
ibi_df   = pd.read_csv(ROOT/"ibi_variability.csv")
train_bpm = pd.read_csv(ROOT/"training_data_bpm_summary.csv").set_index("dataset")

print("=" * 78)
print("PAPER-NUMBERS AUDIT")
print("=" * 78)

# --- §5.1: F-measure by number of active tags --------------------------------
print("\n§5.1  F-measure by number of descriptor tags (paper: 0.732 -> 0.413):")
for n in sorted(res_df["n_descriptors"].unique()):
    sub = res_df[res_df["n_descriptors"] == n]
    if len(sub) >= 3:
        print(f"  {n} tag{'s' if n!=1 else ' '}: n={len(sub):>3}  F = {sub['F-measure'].mean():.3f}")

# --- §5.1: Training-set mean BPM references ---------------------------------
print("\n§5.1  Training-set mean BPMs referenced in paper text:")
for ds in ("ballroom", "hains"):
    if ds in train_bpm.index:
        m = train_bpm.loc[ds, "mean_bpm"]
        print(f"  {ds:<12} mean BPM = {m:.1f}   (paper cites: "
              f"{'Ballroom 130' if ds=='ballroom' else 'Hainsworth 114'})")

# --- Table 2: IBI coefficient of variation (median) -------------------------
print("\nTable 2  Median IBI CV per dataset (paper column):")
cv = ibi_df.groupby("dataset")["cv"].median()
for ds in ["Ballroom", "Beatles", "GTZAN", "Hainsworth", "SMC"]:
    if ds in cv.index:
        print(f"  {ds:<12} CV = {cv[ds]:.3f}")

# --- §5.4.2: Beat Transformer on 45 slow tracks (min_bpm widening) ----------
print("\n§5.4.2  Beat Transformer F on the 45 slow (GT BPM < 55) tracks:")
slow = btc[btc["gt_bpm"] < 55]
print(f"  N slow tracks: {len(slow)}")
print(f"  DBN [55,215] (paper default):   F = {slow['bt_paper_F'].mean():.3f}")
print(f"  DBN [30,215] (wide):            F = {slow['bt_wide_F'].mean():.3f}")
print(f"  Delta from widening: {slow['bt_wide_F'].mean() - slow['bt_paper_F'].mean():+.3f}")

# --- §5.4.3: Beat Transformer tempo-head accuracy by BPM range --------------
print("\n§5.4.3  Beat Transformer tempo head accuracy by BPM range:")
ranges = [(0, 55, "<55 BPM"), (55, 70, "55-70"), (70, 90, "70-90"),
          (90, 120, "90-120"), (120, 1000, ">120 BPM")]
for lo, hi, label in ranges:
    sub = btc[(btc["gt_bpm"] >= lo) & (btc["gt_bpm"] < hi)]
    if len(sub):
        acc = (sub["tempo_class"] == "correct").mean()
        print(f"  {label:<10} n={len(sub):>3}  accuracy = {acc:5.1%}")

# --- §5.4.5: GT-tempo DBN effect stratified by error category ---------------
# Use the priority-ordered (cell 27) taxonomy so numbers align with §5.2.
def _body_cat(row):
    if row["F-measure"] >= 0.8: return "good"
    if row["AMLt"] - row["F-measure"] > 0.25: return "octave_error"
    if row["CMLt"] - row["CMLc"] > 0.2: return "continuity_drift"
    if row["F-measure"] < 0.3 and row["AMLt"] < 0.3: return "total_failure"
    return "other"
_cats = res_df.apply(_body_cat, axis=1)

print("\n§5.4.5  GT-tempo effect stratified by error category (paper §5.4.5):")
print(f"  {'category':<18} {'N':>4} {'raw F':>8} {'GT-t F':>8} {'dF':>7} {'raw CMLt':>9} {'GT-t CMLt':>10} {'dCMLt':>7}")
common = res_df.index.intersection(btc.index)
for cat in ("good", "octave_error", "continuity_drift", "total_failure", "other"):
    mask = (_cats == cat) & _cats.index.isin(common)
    sub_ids = _cats.index[mask]
    if len(sub_ids) == 0:
        continue
    raw_f    = res_df.loc[sub_ids, "F-measure"].mean()
    gt_f     = btc.loc[sub_ids, "btthis_oracle_F"].mean()
    raw_cmlt = res_df.loc[sub_ids, "CMLt"].mean()
    gt_cmlt  = btc.loc[sub_ids, "btthis_oracle_CMLt"].mean()
    print(f"  {cat:<18} {len(sub_ids):>4} {raw_f:>8.3f} {gt_f:>8.3f} "
          f"{gt_f-raw_f:>+7.3f} {raw_cmlt:>9.3f} {gt_cmlt:>10.3f} {gt_cmlt-raw_cmlt:>+7.3f}")

# --- Table 1: ΔCMLt by axis (on / off) -------------------------------------
print("\nTable 1  GT-tempo ΔCMLt (on-axis / off-axis):")
for ax in AXES:
    on = res_df[f"axis:{ax}"] == 1
    common_ax = res_df.index[on].intersection(btc.index)
    common_off = res_df.index[~on].intersection(btc.index)
    dC_on  = (btc.loc[common_ax,  "btthis_oracle_CMLt"] - res_df.loc[common_ax,  "CMLt"]).mean()
    dC_off = (btc.loc[common_off, "btthis_oracle_CMLt"] - res_df.loc[common_off, "CMLt"]).mean()
    print(f"  {ax:<22} on: {dC_on:+.3f}   off: {dC_off:+.3f}")

# --- §6.1: "~100 tracks produce confident activation peaks at wrong positions"
# Proxy count: tracks where bt_act_max >= 0.8 but bt_F < 0.5 (confident but wrong).
_diag = pd.read_csv(ROOT/"activation_diagnostics.csv").set_index("track_id")
_joined = _diag.join(res_df[["F-measure"]], how="inner")
confident_wrong = ((_joined["bt_act_max"] >= 0.8) & (_joined["F-measure"] < 0.5)).sum()
print(f"\n§6.1  Tracks with confident-but-wrong activations (max>=0.8 & F<0.5): {confident_wrong}")
print(f"       (paper cites '~100 tracks')")


# ============================================================================
# EXTENDED AUDIT — every remaining paper number
# ============================================================================

# --- §4: Beat Transformer frame rate (paper: 43.07 FPS) ---------------------
BT_FPS = 44100 / 1024
print(f"\n§4  Beat Transformer FPS: {BT_FPS:.2f}  (paper: 43.07)")

# --- §5.2 / Figure 2: category counts under BOTH taxonomies -----------------
# Paper §5.2 uses a priority-exclusive F-based taxonomy (good=62, ...).
# Figure 2 legend uses OVERLAPPING rule-based counts with a stricter "good"
# (both Beat This AND Beat Transformer F>=0.8) — hence good=39.
_diag = pd.read_csv(ROOT/"activation_diagnostics.csv").set_index("track_id")
_both = res_df[["F-measure","AMLt","CMLt","CMLc"]].join(_diag[["btr_F"]], how="left")

# §5.2 taxonomy (priority-exclusive)
print("\n§5.2  Error taxonomy (priority-exclusive, Beat This F only):")
print(f"  good  (F>=0.8):                       {(_cats == 'good').sum()}  (paper: 62)")
print(f"  octave_error  (AMLt-F > 0.25):        {(_cats == 'octave_error').sum()}  (paper: 17)")
print(f"  continuity_drift  (CMLt-CMLc > 0.2):  {(_cats == 'continuity_drift').sum()}  (paper: 38)")
print(f"  total_failure  (F,AMLt < 0.3):        {(_cats == 'total_failure').sum()}  (paper: 11)")
print(f"  other:                                {(_cats == 'other').sum()}  (paper: 89)")

# Figure 2 legend counts (overlapping rule-based)
print("\nFigure 2  Legend counts (rule-based, can overlap):")
n_good_fig   = ((_both["F-measure"] >= 0.8) & (_both["btr_F"] >= 0.8)).sum()
n_decent_fig = ((_both["F-measure"] >= 0.6) & (_both["F-measure"] < 0.8)).sum()
n_oct_fig    = (_both["AMLt"] - _both["F-measure"] > 0.25).sum()
n_drift_fig  = ((_both["CMLt"] - _both["CMLc"] > 0.2) &
                (_both["F-measure"] >= 0.3) & (_both["F-measure"] <= 0.6)).sum()
n_mod_fig    = ((_both["F-measure"] >= 0.3) & (_both["F-measure"] < 0.6)).sum()
n_fail_fig   = ((_both["F-measure"] < 0.3) & (_both["AMLt"] < 0.3)).sum()
print(f"  good  (BT & BTR F>=0.8):      {n_good_fig}  (paper: 39)")
print(f"  decent  (0.6<=F<0.8):         {n_decent_fig}  (paper: 52)")
print(f"  octave_error:                 {n_oct_fig}  (paper: 16)")
print(f"  continuity_drift  (F 0.3-0.6):{n_drift_fig}  (paper: 14)")
print(f"  moderate  (0.3<=F<0.6):       {n_mod_fig}  (paper: 85)")
print(f"  total_failure:                {n_fail_fig}  (paper: 11)")

# --- §5.1: F-measure by number of active difficulty axes --------------------
print("\n§5.1  F-measure by number of active difficulty axes (paper: 0.786->0.512):")
for k in range(5):
    sub = res_df[res_df["n_axes"] == k]
    if len(sub):
        print(f"  {k} axes: n={len(sub):>3}  F = {sub['F-measure'].mean():.3f}")

# --- Figure 2 caption: smc_274 specific AMLt -------------------------------
row = res_df.loc["smc_274"] if "smc_274" in res_df.index else res_df[res_df.index.astype(str).str.contains("274")].iloc[0]
print(f"\nFigure 2 caption  smc_274: F={row['F-measure']:.3f}  AMLt={row['AMLt']:.3f}  (paper: AMLt=0.988)")

# --- §5.3: total failure wrong-peak vs misaligned breakdown -----------------
# Paper claims "10 wrong peaks, 1 reasonable-but-misaligned".
# The heuristic in scripts/activation_analysis.py classifies 6 strictly as
# wrong_peaks; the 3 borderline cases (gt_peak_mean ~0.34-0.45) are
# functionally wrong peaks too (non-GT activations dominate GT activations).
_fail_ids = _cats.index[_cats == "total_failure"]
_fail_diag = _diag.loc[_fail_ids]
strict_wrong = (_fail_diag["bt_failure_mode"] == "wrong_peaks").sum()
misaligned   = (_fail_diag["bt_failure_mode"] == "reasonable_but_misaligned").sum()
ambiguous    = (_fail_diag["bt_failure_mode"] == "ambiguous_peaks").sum()
other_mode   = (_fail_diag["bt_failure_mode"] == "other").sum()
borderline_wrong = ((_fail_diag["bt_act_nongt_peak_mean"] > _fail_diag["bt_act_gt_peak_mean"]) &
                    (_fail_diag["bt_failure_mode"] != "wrong_peaks") &
                    (_fail_diag["bt_failure_mode"] != "reasonable_but_misaligned")).sum()
print(f"\n§5.3  Total-failure failure-mode breakdown (n={len(_fail_ids)}):")
print(f"  strict wrong_peaks:               {strict_wrong}")
print(f"  reasonable_but_misaligned:        {misaligned}")
print(f"  ambiguous / other (borderline):   {ambiguous + other_mode}  (all have non-GT > GT peaks)")
print(f"  --> functionally wrong peaks:     {strict_wrong + borderline_wrong}   misaligned: {misaligned}")
print(f"  (paper text says '10 wrong peaks, 1 misaligned'; notebook supports 9 wrong, 2 misaligned)")

# --- §5.3.2: synthetic GT Gaussian sigma (paper: sigma=2 frames = 40 ms) ----
BT_THIS_FPS = 50.0
sigma_frames = 2
sigma_ms = 1000 * sigma_frames / BT_THIS_FPS
print(f"\n§5.3.2  Synthetic GT Gaussian: sigma = {sigma_frames} frames "
      f"= {sigma_ms:.0f} ms at {BT_THIS_FPS:.0f} FPS  (Beat This activation rate)")

# --- §5.4.1: DBN worse/improved/unchanged on 217 tracks ---------------------
_d = res_df["dbn_F-measure"] - res_df["F-measure"]
worse_strict   = (_d < 0).sum()
improved_strict = (_d > 0).sum()
equal_strict    = (_d == 0).sum()
print(f"\n§5.4.1  DBN impact on Beat This activations (paper: 124/54/39):")
print(f"  AMLt change: {res_df['dbn_AMLt'].mean() - res_df['AMLt'].mean():+.3f}  "
      f"({res_df['AMLt'].mean():.3f} -> {res_df['dbn_AMLt'].mean():.3f})  (paper: +0.044 -> 0.654)")
print(f"  Strict   (>, <, ==):       worsened={worse_strict}, improved={improved_strict}, unchanged={equal_strict}")
worse_tol = (_d < -0.01).sum(); improved_tol = (_d > 0.01).sum(); tol_equal = len(_d) - worse_tol - improved_tol
print(f"  +/-0.01 tolerance:         worsened={worse_tol}, improved={improved_tol}, unchanged={tol_equal}")

# Non-tempo-instability subset (for the paper's "-0.012, 29% worsened" claim)
ti_off = res_df[res_df["axis:Tempo instability"] == 0]
d_off = ti_off["dbn_F-measure"] - ti_off["F-measure"]
print(f"  Non-tempo-instability (N={len(ti_off)}): dF = {d_off.mean():+.3f}, "
      f"%hurt(>0.01) = {100*(d_off < -0.01).mean():.1f}%  (paper: -0.012, 29%)")

# --- §5.4.5: smc_064 specific GT-tempo values -------------------------------
_gt = pd.read_csv(ROOT/"gt_tempo_lambda_sweep_results.csv").set_index("track_id")
if "smc_064" in _gt.index:
    r = _gt.loc["smc_064"]
    print(f"\n§5.4.5  smc_064 under GT-tempo DBN (paper: F=0.000, CMLt=0.880):")
    print(f"  default lambda=100: F={r['default_lambda_F']:.3f}  CMLt={r['default_lambda_CMLt']:.3f}")
    print(f"  optimal lambda:     F={r['optimal_F']:.3f}  CMLt={r['optimal_CMLt']:.3f}  (lambda={int(r['optimal_lambda'])})")

# --- §5.4.5: gap decomposition 0.297 = 0.251 (activation) + 0.046 (decoding)
_gt_act_F   = 0.924                                 # synthetic GT+DBN (from cell 46)
_raw_peak_F = res_df["F-measure"].mean()            # Beat This raw peak-picking
_decoder_ceiling_F = 0.673                          # per-track optimal threshold (cell 45)
total_gap   = _gt_act_F - _raw_peak_F
decoder_gap = _decoder_ceiling_F - _raw_peak_F
activation_gap = _gt_act_F - _decoder_ceiling_F
print(f"\n§5.4.5  Activation-vs-decoding gap decomposition:")
print(f"  GT activations + DBN:                     F = {_gt_act_F:.3f}")
print(f"  Per-track optimal-threshold ceiling:      F = {_decoder_ceiling_F:.3f}")
print(f"  Raw peak-picking baseline:                F = {_raw_peak_F:.3f}")
print(f"  Total gap:        {total_gap:.3f}")
print(f"  Activation gap:   {activation_gap:.3f}  ({100*activation_gap/total_gap:.0f}%)  <-- dominant")
print(f"  Decoding gap:     {decoder_gap:.3f}  ({100*decoder_gap/total_gap:.0f}%)")
print(f"  (paper: 85% activation (F=0.251), 15% decoding (F=0.046))")

# --- §5.4.6: median optimal lambda by error category ------------------------
_lam = pd.read_csv(ROOT/"transition_lambda_sweep_results.csv").set_index("track_id")
print(f"\n§5.4.6  Median optimal lambda by §5.2 error category:")
print(f"  {'category':<18} {'N':>4} {'median lambda':>14} {'mean lambda':>13}")
for cat in ("good", "octave_error", "continuity_drift", "total_failure", "other"):
    mask = _cats == cat
    ids = _cats.index[mask].intersection(_lam.index)
    if len(ids) == 0:
        continue
    med = _lam.loc[ids, "optimal_lambda"].median()
    mn  = _lam.loc[ids, "optimal_lambda"].mean()
    print(f"  {cat:<18} {len(ids):>4} {med:>14.1f} {mn:>13.2f}")
print(f"  (paper: good median ~2, total_failure ~30, octave_error ~20)")

# --- §6.1: activation + tempo ceilings (already printed by cell 81) ---------
# Repeat here so the audit section is self-contained.
_gt_lam = pd.read_csv(ROOT/"gt_tempo_lambda_sweep_results.csv")
_tl     = pd.read_csv(ROOT/"transition_lambda_sweep_results.csv")
print(f"\n§6.1  Two performance ceilings (paper: F ~ 0.67, CMLt ~ 0.70, AMLt ~ 0.73):")
print(f"  Activation ceiling (wide DBN + per-track optimal lambda): F = {_tl['optimal_F'].mean():.4f}")
print(f"  GT-tempo DBN + optimal lambda:                             F = {_gt_lam['optimal_F'].mean():.4f}   "
      f"CMLt = {_gt_lam['optimal_CMLt'].mean():.4f}   AMLt = {_gt_lam['optimal_AMLt'].mean():.4f}")

# --- §6.1: "~100 tracks" confident-but-wrong activations --------------------
# Report multiple reasonable criteria — the paper's "~100" is closest to the
# (F<0.8 AND act@GT < 0.7) rule, which captures both weak-at-GT and
# high-max-elsewhere patterns.
_dj = _diag.join(res_df[["F-measure"]], how="inner")
c1 = ((_dj["bt_act_max"] >= 0.8) & (_dj["F-measure"] < 0.5)).sum()
c2 = ((_dj["F-measure"] < 0.8) & (_dj["bt_act_gt_peak_mean"] < 0.7)).sum()
c3 = ((_dj["F-measure"] < 0.8) & (_dj["bt_act_nongt_peak_mean"] > _dj["bt_act_gt_peak_mean"])).sum()
print(f"\n§6.1  Confident-but-wrong activation tracks (paper cites '~100'):")
print(f"  max_act >= 0.8 AND F < 0.5:                  n = {c1}")
print(f"  F < 0.8 AND act@GT < 0.7:                    n = {c2}   <-- closest to '~100'")
print(f"  F < 0.8 AND nongt_peak > gt_peak:            n = {c3}")
